# PTD-corpus — Planos de Transformação Digital (Gov.br)

Notebook de **engenharia do corpus** dos Planos de Transformação Digital dos
órgãos federais brasileiros: coleta, extração, padronização e exportação.

Spin-off do projeto PTD **sem a camada de analytics** (estatísticas, figuras,
dashboard, insumos da nota técnica) — ver `HANDOUT.md`.

**Fonte:** [gov.br/governodigital - Planos de Transformação Digital](https://www.gov.br/governodigital/pt-br/estrategias-e-governanca-digital/planos-de-transformacao-digital)

**Pipeline:**
1. Scraping da lista de órgãos signatários e URLs dos PDFs
2. Download dos PDFs (Documento Diretivo + Anexo de Entregas)
3. Extração de tabelas (PyMuPDF `find_tables()`)
4. Padronização de vocabulário
5. Exportação CSV/JSON + relatório de erros
6. Validação com checksums + bundle de publicação

In [ ]:
# ============================================================
# CÉLULA 1 — Setup & Instalação
# ============================================================
# Detecta ambiente (Google Colab ou local) e instala dependências.
# Execute esta célula primeiro.

import os, sys

# --- Detecção de ambiente ---
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Instalação de pacotes (apenas no Colab; local usa requirements.txt) ---
# IMPORTANTE: manter alinhado com requirements.txt na raiz do repo.
if IN_COLAB:
    get_ipython().system('pip install -q pymupdf pypdf beautifulsoup4 requests tqdm pandas')

# --- Google Drive (persistência de PDFs e outputs no Colab) ---
USE_DRIVE = IN_COLAB

if USE_DRIVE:
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/PTD_Scraper"
else:
    BASE_DIR = os.path.join(os.getcwd(), "ptd_output")

DIRS = {
    "pdfs_diretivo":  os.path.join(BASE_DIR, "pdfs", "diretivo"),
    "pdfs_entregas":  os.path.join(BASE_DIR, "pdfs", "entregas"),
    "output":         os.path.join(BASE_DIR, "output"),
    "checkpoints":    os.path.join(BASE_DIR, "checkpoints"),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print(f"Ambiente: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Diretório base: {BASE_DIR}")
print("Estrutura criada:", list(DIRS.keys()))

In [ ]:
# ============================================================
# CÉLULA 2 — Configuração, Constantes e Estruturas de Dados
# ============================================================
import os, sys, time, pickle, unicodedata, re, json, logging, difflib
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Tuple, Dict, Any
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# --------------- URLs e parâmetros de rede ------------------
BASE_URL = ("https://www.gov.br/governodigital/pt-br/"
            "estrategias-e-governanca-digital/"
            "planos-de-transformacao-digital")
REQUEST_DELAY = 2.0        # segundos entre requests ao gov.br
MAX_RETRIES   = 4
REQUEST_TIMEOUT = 90
HTTP_HEADERS  = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/124.0 Safari/537.36"),
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.5",
}

# --------------- Vocabulários canônicos ---------------------
CANONICAL_EIXOS = [
    "Serviços Digitais e Melhoria da Qualidade",
    "Unificação de Canais Digitais",
    "Governança e Gestão de Dados",
    "Segurança e Privacidade",
    "Projetos Especiais",
]

# Mapa Eixo → lista de Produtos (44 produtos oficiais)
CANONICAL_PRODUTOS: Dict[str, List[str]] = {
    "Serviços Digitais e Melhoria da Qualidade": [
        "Disponibilização em Acesso Digital",
        "Disponibilização de datas/cronograma na Agenda Gov.Br",
        "Evolução do Serviço",
        "Implantação da Área Logada Gov.Br",
        "Implantação da Experiência LabQ",
        "Implementação do VLIBRAS",
        "Integração à ferramenta de acompanhamento das solicitações",
        "Integração à ferramenta de avaliação da satisfação dos usuários",
        "Realização de Autodiagnóstico de Qualidade",
        "Revisão da descrição dos serviços",
        "Oferta de agendamento digital",
        "Integração à ferramenta de solicitação de atendimento presencial",
        "Implantação do Atendimento Virtual",
    ],
    "Unificação de Canais Digitais": [
        "Implantação do Design System",
        "Integração ao Login Único",
        "Integração ao Pagtesouro",
        "Integração com Assinatura Digital Gov.br",
        "Migração de APPs móveis para a loja do Gov.br",
        "Migração de Portal Institucional",
        "Migração de Serviço para Plataforma Unificada",
    ],
    "Governança e Gestão de Dados": [
        "Abertura de bases de dados",
        "Adequação à LGPD",
        "Catalogação de bases de dados no Portal de Dados",
        "Criação de Comitê de Governança de Dados",
        "Elaboração de Plano de Dados Abertos",
        "Elaboração do PDTIC",
        "Elaboração/Revisão da POSIC",
        "Implantação de processo de gestão de dados",
        "Implantação do inventário de bases de dados",
        "Implementação da interoperabilidade de dados",
        "Integração de dados ao Conecta Gov.br",
        "Integração de dados ao Datamart",
        "Melhoria da qualidade de bases de dados",
        "Nomeação de Encarregado de Dados Pessoais",
        "Obtenção de certificação de bases de dados",
        "Publicação de dados no Portal de Dados Abertos",
        "Realização de Inventário de Dados",
        "Relatório de Impacto à Proteção de Dados Pessoais",
        "Resposta a demandas de compartilhamento",
        "Utilização de dados do Conecta Gov.br",
    ],
    "Segurança e Privacidade": [
        "Adequação ao Framework de Privacidade e Segurança da Informação",
        "Elevação do nível de maturidade em privacidade e segurança",
    ],
    "Projetos Especiais": [
        "Iniciativa de Transformação Digital",
        "Projeto Especial de Transformação Digital",
    ],
}

# ---------- Produtos de templates anteriores (v1.x, v2.x) -----
# Produtos que aparecem em PTDs mais antigos e não estão no template v4.0
# Mapeados ao canônico mais próximo ou mantidos como produto válido extra
LEGACY_PRODUTOS: Dict[str, List[str]] = {
    "Segurança e Privacidade": [
        "Implementação do PPSI",                             # template v2.x
        "Adequação ao PPSI",                                 # variante
        "Auto-avaliação, análise de lacunas e planejamento do PPSI",
    ],
    "Governança e Gestão de Dados": [
        "Integração à base de dados",                        # template v2.x genérico
        "Interoperabilidade de Sistemas",                    # eixo antigo EGD 2020
        "Compartilhamento de dados via Conecta Gov.br",      # variante
    ],
    "Serviços Digitais e Melhoria da Qualidade": [
        "Digitalização de Serviço",                          # EGD 2020-2022
        "Publicação de Serviço no Portal Gov.br",            # EGD 2020-2022
        "Transformação Digital de Serviço",                   # EGD 2020-2022
    ],
    "Projetos Especiais": [
        "Ação estratégica de transformação digital",         # variante
        "Outros",                                            # produto genérico usado por 39 órgãos
    ],
}

# ---------- Aliases: texto variante → canônico exato ----------
# Mapeamento determinístico de variações conhecidas
PRODUTO_ALIASES: Dict[str, str] = {
    # Truncamentos e variações de acentuação
    "Integração à ferramenta de avaliação da satisfação dos usuários dos serviços":
        "Integração à ferramenta de avaliação da satisfação dos usuários",
    "Evolução do Serviço Digital":
        "Evolução do Serviço",
    "Integração ao Login Único Gov.Br":
        "Integração ao Login Único",
    "Integração ao Login Unico":
        "Integração ao Login Único",
    "Implementação do VLibras":
        "Implementação do VLIBRAS",
    "Implementacao do VLIBRAS":
        "Implementação do VLIBRAS",
    "Implantação da Area Logada Gov.Br":
        "Implantação da Área Logada Gov.Br",
    "Migração de Serviço para Plataforma Unificada Gov.br":
        "Migração de Serviço para Plataforma Unificada",
    "Migração do Portal Institucional":
        "Migração de Portal Institucional",
    "Adequação à Lei Geral de Proteção de Dados":
        "Adequação à LGPD",
    "Elaboração do Plano Diretor de TIC":
        "Elaboração do PDTIC",
    "Elaboração da POSIC":
        "Elaboração/Revisão da POSIC",
    "Revisão da POSIC":
        "Elaboração/Revisão da POSIC",
    "Integração à ferramenta de acompanhamento de solicitações":
        "Integração à ferramenta de acompanhamento das solicitações",
    "Disponibilização em acesso digital":
        "Disponibilização em Acesso Digital",
    "Revisão da descrição de serviços":
        "Revisão da descrição dos serviços",
    # Variantes ortográficas observadas em PDFs gov.br 2024-2026
    "Autoavaliação, análise de lacunas e planejamento do PPSI":
        "Auto-avaliação, análise de lacunas e planejamento do PPSI",
    # Truncamentos de célula PDF (largura fixa corta o fim do texto) —
    # observados no snapshot 2026-05; cobrem ~430 dos 531 fuzzy_high.
    "Integração à ferramenta de avaliação da":
        "Integração à ferramenta de avaliação da satisfação dos usuários",
    "Integração à base de dados (outros)":
        "Integração à base de dados",
    "Auto-avaliação, análise de lacunas e pla":
        "Auto-avaliação, análise de lacunas e planejamento do PPSI",
    "Migração de Serviço para Plataforma":
        "Migração de Serviço para Plataforma Unificada",
    "Disponibilização em Acesso D":
        "Disponibilização em Acesso Digital",
    "Implantação da Área Logada":
        "Implantação da Área Logada Gov.Br",
    # Artefato de quebra de linha: caractere da célula vizinha colado no
    # início do produto (normalize_text não remove letra solta inicial).
    "eIntegração ao Login Único":
        "Integração ao Login Único",
    "eDisponibilização em Acesso Digital":
        "Disponibilização em Acesso Digital",
    "eIntegração à ferramenta de avaliação da":
        "Integração à ferramenta de avaliação da satisfação dos usuários",
    # Legacy / EGD 2020 mappings
    "Digitalização de Serviço":
        "Disponibilização em Acesso Digital",
    "Publicação de Serviço no Portal Gov.br":
        "Disponibilização em Acesso Digital",
    "Transformação Digital de Serviço":
        "Disponibilização em Acesso Digital",
}

# ---------- Eixos legados (EGD 2020-2022) → canônico ----------
EIXO_ALIASES: Dict[str, str] = {
    "Transformação Digital de Serviços Públicos":
        "Serviços Digitais e Melhoria da Qualidade",
    "Transformação Digital dos Serviços":
        "Serviços Digitais e Melhoria da Qualidade",
    "Unificação de Canais Digitais e Plataformas":
        "Unificação de Canais Digitais",
    "Governo como Plataforma":
        "Unificação de Canais Digitais",
    "Governo Aberto e Transparência":
        "Governança e Gestão de Dados",
    "Infraestrutura de TIC e Governança de Dados":
        "Governança e Gestão de Dados",
    "Interoperabilidade de Sistemas":
        "Governança e Gestão de Dados",
    "Dados para o Desenvolvimento":
        "Governança e Gestão de Dados",
    "Identidade Digital do Cidadão":
        "Unificação de Canais Digitais",
    "Governo Inteligente":
        "Projetos Especiais",
    # Typo observado em PDF do PGFN
    "Governaça e Gestão de Dados":
        "Governança e Gestão de Dados",
    # Artefatos de extração PDF: início truncado ('Ser' cortado) e/ou
    # caractere da célula vizinha colado (quebra de linha).
    "iviços Digitais e Melhoria da Qualidade":
        "Serviços Digitais e Melhoria da Qualidade",
    "iServiços Digitais e Melhoria da Qualidade":
        "Serviços Digitais e Melhoria da Qualidade",
    "pSegurança e Privacidade":
        "Segurança e Privacidade",
}

# Lista flat de todos os produtos (canônicos + legados)
ALL_PRODUTOS = [p for prods in CANONICAL_PRODUTOS.values() for p in prods]
ALL_PRODUTOS += [p for prods in LEGACY_PRODUTOS.values() for p in prods]

# Mapa reverso: produto → eixo canônico (canônicos + legados)
PRODUTO_TO_EIXO = {}
for eixo, prods in CANONICAL_PRODUTOS.items():
    for p in prods:
        PRODUTO_TO_EIXO[p] = eixo
for eixo, prods in LEGACY_PRODUTOS.items():
    for p in prods:
        PRODUTO_TO_EIXO[p] = eixo

# Thresholds de qualidade — usados por 13b (validation_report) + run_pipeline.py para detectar regressões
# (ex: dedup pulado, checkpoint stale carregado, novo formato de PDF não suportado).
# Bumpar conforme o corpus crescer ou o gov.br republicar com novos rótulos.
QUALITY_THRESHOLDS = {
    "max_entregas":               4700,   # 4574 baseline + margem para PDFs novos
    "max_riscos":                  700,   # 619 baseline + margem
    "min_prob_canonica_ratio":    0.85,
    "min_imp_canonica_ratio":     0.85,
    "min_trat_canonica_ratio":    0.80,
}

# Escalas do Documento Diretivo (tabela de riscos)
PROBABILIDADE_SCALE = [
    "raro", "pouco provável", "provável",
    "muito provável", "praticamente certo",
]
IMPACTO_SCALE = [
    "muito baixo", "baixo", "médio", "alto", "muito alto",
]
TRATAMENTO_OPTIONS = ["mitigar", "eliminar", "transferir", "aceitar"]

# Aliases de escala — mapeamento semântico de variações metodológicas
# usadas por alguns órgãos para os 5 níveis canônicos da SGD.
# - ANTAQ usa 3 pontos (Baixa/Média/Alta) → comprime para níveis intermediários
# - SUSEP usa numérica 1-4 (omite o 5° nível) → aproxima ao final da escala
# - CADE mistura numerada com label (1-Alto, 2-Alto)
PROBABILIDADE_ALIASES: Dict[str, str] = {
    # ANTAQ 3-pontos
    "muito baixa": "raro",
    "baixa": "pouco provável",
    "media": "provável",
    "média": "provável",
    "alta": "muito provável",
    "muito alta": "praticamente certo",
    # SUSEP numérica
    "1": "raro",
    "2": "pouco provável",
    "3": "provável",
    "4": "muito provável",
    "5": "praticamente certo",
    # Variantes lexicais
    "rara": "raro",
    "raros": "raro",
    "raras": "raro",
    "praticamente certa": "praticamente certo",
    # Observadas em PDFs gov.br 2024-2026
    "pouca": "pouco provável",
    "certo": "praticamente certo",
    "baixo": "pouco provável",       # variante masculina ocasional
    "provavel muito": "muito provável",   # MMULHERES — ordem PT invertida
    "provavel pouco": "pouco provável",   # MMULHERES — idem
}
IMPACTO_ALIASES: Dict[str, str] = {
    # ANTAQ
    "grande": "muito alto",
    "moderado": "médio",
    "moderada": "médio",
    # CADE com prefixo numérico
    "1-alto": "alto",
    "2-alto": "alto",
    "1-medio": "médio",
    "1-médio": "médio",
    "2-medio": "médio",
    "2-médio": "médio",
    # SUSEP numérica (mesmo mapping da probabilidade)
    "1": "muito baixo",
    "2": "baixo",
    "3": "médio",
    "4": "alto",
    "5": "muito alto",
    # Variantes lexicais
    "baixa": "baixo",
    "media": "médio",
    "média": "médio",
    "alta": "alto",
    "muito baixa": "muito baixo",
    "muito alta": "muito alto",
    # Observadas em PDFs gov.br 2024-2026
    "crítico": "muito alto",
    "critico": "muito alto",
    "alto muito": "muito alto",       # ordem trocada por quebra de linha
    "muito alto alto": "muito alto",  # duplicação por quebra de linha
    "medio muito": "médio",           # MMULHERES — ordem trocada por quebra de linha
}
TRATAMENTO_ALIASES: Dict[str, str] = {
    "mitigar o risco": "mitigar",
    "reduzir": "mitigar",
    "reduzir ou mitigar": "mitigar",
    "reduzir ou mitigar o risco": "mitigar",
    "tratar": "mitigar",
    "monitorar": "mitigar",
    "tolerar": "aceitar",
    "aceitar ou tolerar": "aceitar",
    "aceitar ou tolerar o risco": "aceitar",
    "compartilhar": "transferir",
    "compartilhar o risco": "transferir",
    "compartilh ar": "transferir",   # quebra de linha intra-palavra
    "evitar": "eliminar",
    "mitigar/transferir": "mitigar; transferir",  # AEB — separador "/"
    "transferir/compartilhar": "transferir",      # ANA — variante "/"
    "3-mitigar": "mitigar",                       # CADE — prefixo numérico
}

# ---------- Órgãos que compartilham PDFs (grupos) ----------
ORGAN_GROUPS: Dict[str, List[str]] = {
    "MD":   ["MD", "CEX", "CM", "COMAER", "CENSIPAM", "FOSORIO", "HFA"],
    "MEC":  ["MEC", "CAPES", "EBSERH", "FNDE", "FUNDAJ", "IBC", "INEP", "INES"],
    "MF":   ["MF", "RFB", "STN", "PGFN"],
    "MMA":  ["MMA", "IBAMA", "ICMBIO", "SFB", "JBRJ"],
    "MT":   ["MT", "ANTT", "DNIT"],
    "MIDR": ["MIDR", "CODEVASF", "SUDAM", "SUDECO", "SUDENE"],
    "MDA":  ["MDA", "CONAB"],
}
# Reverso: sigla membro → sigla cabeça do grupo
MEMBER_TO_GROUP = {}
for head, members in ORGAN_GROUPS.items():
    for m in members:
        if m != head:
            MEMBER_TO_GROUP[m] = head

# --------------- Dataclasses --------------------------------
@dataclass
class OrganInfo:
    sigla: str
    nome_completo: str
    grupo: Optional[str] = None          # sigla do cabeça, se agrupado
    url_diretivo: Optional[str] = None
    url_entregas: Optional[str] = None
    pdf_path_diretivo: Optional[str] = None
    pdf_path_entregas: Optional[str] = None

@dataclass
class RiskEntry:
    orgao_sigla: str
    risco_texto: str = ""
    probabilidade_original: str = ""
    probabilidade_normalizada: str = ""
    probabilidade_score: float = 0.0
    probabilidade_method: str = ""    # exact | alias | fuzzy_high | fuzzy_low | unmatched
    impacto_original: str = ""
    impacto_normalizado: str = ""
    impacto_score: float = 0.0
    impacto_method: str = ""
    tratamento_original: str = ""
    tratamento_normalizado: str = ""
    tratamento_score: float = 0.0
    tratamento_method: str = ""
    acoes_tratamento: str = ""
    extraction_confidence: str = "high"    # high / medium / low
    needs_review: bool = False
    review_reason: Optional[str] = None

@dataclass
class DeliveryEntry:
    orgao_sigla: str
    tabela_tipo: str = "pactuada"          # default; "concluida" / "cancelada" quando os PTDs publicarem ciclos futuros
    servico_acao: str = ""
    produto_original: str = ""
    produto_normalizado: str = ""
    produto_score: float = 0.0
    produto_method: str = ""               # exact | alias | fuzzy_high | fuzzy_low | unmatched
    eixo_original: str = ""
    eixo_normalizado: str = ""
    eixo_score: float = 0.0
    eixo_method: str = ""
    area_responsavel: Optional[str] = None
    data_pactuada: Optional[str] = None
    data_entrega: Optional[str] = None
    pactuado: Optional[str] = None         # Sim / Não (concluídas)
    justificativa: Optional[str] = None    # (canceladas)
    extraction_confidence: str = "high"
    needs_review: bool = False
    review_reason: Optional[str] = None

@dataclass
class ProcessingError:
    orgao_sigla: str
    document_type: str     # diretivo / entregas
    stage: str             # download / extraction / standardization
    error_type: str
    error_message: str
    timestamp: str = ""
    url: Optional[str] = None

    def __post_init__(self):
        if not self.timestamp:
            self.timestamp = datetime.now().isoformat()

# Contêineres globais
all_organs: List[OrganInfo] = []
all_risks: List[RiskEntry] = []
all_deliveries: List[DeliveryEntry] = []
all_errors: List[ProcessingError] = []

print(f"Configuração carregada: {len(ALL_PRODUTOS)} produtos canônicos em {len(CANONICAL_EIXOS)} eixos")

In [ ]:
# ============================================================
# CÉLULA 3 — Funções Utilitárias
# ============================================================

# --------------- Rede com retry -----------------------------
def safe_request(url: str, max_retries: int = MAX_RETRIES,
                 delay: float = REQUEST_DELAY,
                 timeout: int = REQUEST_TIMEOUT) -> Optional[requests.Response]:
    """GET com retry exponencial e rate-limiting."""
    for attempt in range(1, max_retries + 1):
        try:
            time.sleep(delay)
            resp = requests.get(url, headers=HTTP_HEADERS, timeout=timeout)
            if resp.status_code == 200:
                return resp
            if resp.status_code == 503 and attempt < max_retries:
                wait = delay * (2 ** attempt)
                print(f"  503 em {url} — retry {attempt}/{max_retries} em {wait:.0f}s")
                time.sleep(wait)
                continue
            resp.raise_for_status()
        except requests.RequestException as exc:
            if attempt < max_retries:
                wait = delay * (2 ** attempt)
                print(f"  Erro ({exc}) — retry {attempt}/{max_retries} em {wait:.0f}s")
                time.sleep(wait)
            else:
                print(f"  FALHA definitiva: {url} — {exc}")
                return None
    return None

# --------------- Normalização de texto ----------------------
# Prefixo enumerativo de listas: "A) ", "1) ", "i) ", "II. ", "a- " etc.
# Stripa o marcador para que aliases (e.g., "reduzir ou mitigar") casem
# mesmo quando o PDF do órgão prefixa com letra/número (e.g. CADE: "B) Reduzir...").
_ENUM_PREFIX = re.compile(r"^[A-Za-z0-9]{1,3}\s*[\)\.\-]\s+")

def normalize_text(text: str) -> str:
    """Normaliza unicode, whitespace e caixa para comparação."""
    if not text:
        return ""
    text = str(text)
    # Strip Unicode Cf (Format): ZWSP, ZWJ, soft hyphen, BOM, word joiner,
    # variation selectors, etc. Esses chars vêm de extração PyMuPDF de PDFs
    # com kerning custom e quebram fuzzy_match silenciosamente — strings
    # visualmente idênticas comparam como diferentes, devolvendo ~0.98 via
    # SequenceMatcher em vez de exact match 1.0.
    text = "".join(c for c in text if unicodedata.category(c) != "Cf")
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = _ENUM_PREFIX.sub("", text)
    return text

def strip_accents(text: str) -> str:
    """Remove acentos para matching fuzzy."""
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

# --------------- Matching fuzzy de vocabulário --------------
def fuzzy_match(original: str, candidates: list,
                threshold: float = 0.85) -> Tuple[str, float]:
    """Retorna (melhor_candidato, score). Score 0 se abaixo do threshold."""
    if not original or not candidates:
        return ("", 0.0)
    norm = normalize_text(original).lower()
    norm_stripped = strip_accents(norm)
    # Tentativa exata (case-insensitive, accent-insensitive)
    for c in candidates:
        c_norm = normalize_text(c).lower()
        if norm == c_norm or norm_stripped == strip_accents(c_norm):
            return (c, 1.0)
    # Fuzzy
    best, best_score = "", 0.0
    for c in candidates:
        c_norm = normalize_text(c).lower()
        score = difflib.SequenceMatcher(None, norm_stripped,
                                        strip_accents(c_norm)).ratio()
        if score > best_score:
            best, best_score = c, score
    if best_score >= threshold:
        return (best, best_score)
    return (best, best_score)   # retorna melhor mesmo abaixo do threshold

def fuzzy_match_produto(original: str) -> Tuple[str, float]:
    """Match produto com: aliases determinísticos → canônicos+legados → fuzzy."""
    if not original:
        return ("", 0.0)
    norm = normalize_text(original)
    # Camada 0: alias determinístico (variações conhecidas)
    for alias_key, canonical_val in PRODUTO_ALIASES.items():
        if normalize_text(alias_key).lower() == norm.lower():
            return (canonical_val, 1.0)
        if strip_accents(normalize_text(alias_key).lower()) == strip_accents(norm.lower()):
            return (canonical_val, 0.98)
    # Camada 1+: match fuzzy contra lista completa (canônicos + legados)
    return fuzzy_match(original, ALL_PRODUTOS, threshold=0.80)

def fuzzy_match_eixo(original: str) -> Tuple[str, float]:
    """Match eixo com: aliases legados → canônicos → fuzzy."""
    if not original:
        return ("", 0.0)
    norm = normalize_text(original)
    # Camada 0: alias legado (eixos EGD 2020-2022 → EFGD 2024)
    for alias_key, canonical_val in EIXO_ALIASES.items():
        if normalize_text(alias_key).lower() == norm.lower():
            return (canonical_val, 0.95)
        if strip_accents(normalize_text(alias_key).lower()) == strip_accents(norm.lower()):
            return (canonical_val, 0.93)
    return fuzzy_match(original, CANONICAL_EIXOS, threshold=0.80)

def classify_match(original: str, score: float, alias_map: Optional[dict] = None,
                   fuzzy_high_cut: float = 0.85, fuzzy_low_cut: float = 0.70) -> str:
    """Classifica o resultado de um fuzzy_match em 5 buckets fixos.

    Sempre retorna um dos 5 valores; nunca cresce em cardinalidade.
    Usado para popular RiskEntry.*_method e DeliveryEntry.*_method.

    `alias_map` pode ter keys de duas formas:
    - normalizadas (lowercase + sem accent) — caso de PROBABILIDADE_ALIASES,
      IMPACTO_ALIASES, TRATAMENTO_ALIASES.
    - preservando case+accent — caso de PRODUTO_ALIASES, EIXO_ALIASES.

    Para detectar 'alias' nos dois casos, normaliza-se cada key também.
    Cacheia o set normalizado por id(alias_map) para amortizar custo entre
    chamadas (alias_maps são módulo-level, vivem o run inteiro).
    """
    if not original or score <= 0.0:
        return "unmatched"
    if score >= 0.999:
        return "exact"
    if alias_map:
        norm = strip_accents(normalize_text(original).lower().strip())
        cache_key = id(alias_map)
        if cache_key not in _ALIAS_KEY_NORM_CACHE:
            _ALIAS_KEY_NORM_CACHE[cache_key] = {
                strip_accents(normalize_text(k).lower().strip()) for k in alias_map
            }
        if norm in _ALIAS_KEY_NORM_CACHE[cache_key]:
            return "alias"
    if score >= fuzzy_high_cut:
        return "fuzzy_high"
    if score >= fuzzy_low_cut:
        return "fuzzy_low"
    return "unmatched"


# Cache de keys normalizadas dos alias_maps. Populado lazily em classify_match.
# As keys do dict são id(alias_map); values são set[str] de keys normalizadas.
_ALIAS_KEY_NORM_CACHE: Dict[int, set] = {}


def fuzzy_match_scale(original: str, scale: list) -> Tuple[str, float]:
    """Canoniza valores de escala (probabilidade/impacto/tratamento) com
    suporte a escalas alternativas usadas por alguns órgãos (ANTAQ 3-pontos,
    SUSEP numérica, CADE mista). Aliases têm prioridade sobre fuzzy match."""
    if not original:
        return ("", 0.0)
    norm = strip_accents(normalize_text(original).lower().strip())
    if scale is PROBABILIDADE_SCALE and norm in PROBABILIDADE_ALIASES:
        return (PROBABILIDADE_ALIASES[norm], 0.95)
    if scale is IMPACTO_SCALE and norm in IMPACTO_ALIASES:
        return (IMPACTO_ALIASES[norm], 0.95)
    if scale is TRATAMENTO_OPTIONS and norm in TRATAMENTO_ALIASES:
        return (TRATAMENTO_ALIASES[norm], 0.95)
    return fuzzy_match(original, scale, threshold=0.70)

# --------------- Parse de datas variadas --------------------
_DATE_PATTERNS = [
    (r"(\d{2})/(\d{2})/(\d{4})", lambda m: f"{m.group(3)}-{m.group(2)}-{m.group(1)}"),
    (r"(\d{2})/(\d{4})",          lambda m: f"{m.group(2)}-{m.group(1)}"),
    (r"(\d{4})-(\d{2})-(\d{2})",  lambda m: m.group(0)),
]
_MONTH_MAP = {
    "jan": "01", "fev": "02", "mar": "03", "abr": "04",
    "mai": "05", "jun": "06", "jul": "07", "ago": "08",
    "set": "09", "out": "10", "nov": "11", "dez": "12",
}

def parse_date(text: str) -> Optional[str]:
    """Converte formatos variados para YYYY-MM ou YYYY-MM-DD."""
    if not text:
        return None
    text = normalize_text(text).strip()
    for pat, fmt in _DATE_PATTERNS:
        m = re.search(pat, text)
        if m:
            return fmt(m)
    # Formato "jun/25", "mar/2026"
    m = re.match(r"([a-záéíóú]{3})[\./\-](\d{2,4})", text.lower())
    if m:
        month = _MONTH_MAP.get(m.group(1)[:3])
        year = m.group(2)
        if len(year) == 2:
            year = "20" + year
        if month:
            return f"{year}-{month}"
    return text   # retorna original se não parsear

# --------------- Checkpoint / Resume ------------------------
# Fingerprint sidecar: detecta quando o estado upstream mudou (ex: 05c_dedup.py
# zerou pdf_path de N órgãos) e invalida automaticamente o pickle, em vez de
# carregar dados estagnados que não refletem o pipeline atual.

def state_fingerprint(state: Any) -> str:
    """SHA-1 truncado de uma representação estável de `state`. Use uma estrutura
    pequena e ordenada (ex: lista de tuplas) para que o fingerprint seja
    determinístico entre runs."""
    import hashlib
    payload = repr(state).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()[:12]

def save_checkpoint(data: Any, name: str, fingerprint: Optional[str] = None) -> None:
    pkl_path = os.path.join(DIRS["checkpoints"], f"{name}.pkl")
    with open(pkl_path, "wb") as f:
        pickle.dump(data, f)
    if fingerprint:
        fp_path = os.path.join(DIRS["checkpoints"], f"{name}.fingerprint")
        with open(fp_path, "w") as f:
            f.write(fingerprint)
    print(f"  Checkpoint salvo: {name}" + (f" [fp={fingerprint}]" if fingerprint else ""))

def load_checkpoint(name: str, expected_fingerprint: Optional[str] = None) -> Optional[Any]:
    pkl_path = os.path.join(DIRS["checkpoints"], f"{name}.pkl")
    fp_path = os.path.join(DIRS["checkpoints"], f"{name}.fingerprint")
    if not os.path.exists(pkl_path):
        return None
    if expected_fingerprint is not None:
        actual = ""
        if os.path.exists(fp_path):
            with open(fp_path) as f:
                actual = f.read().strip()
        if actual != expected_fingerprint:
            print(f"  Checkpoint {name}: fingerprint divergente "
                  f"({actual or 'ausente'} != {expected_fingerprint}) — invalidando")
            try:
                os.remove(pkl_path)
                if os.path.exists(fp_path):
                    os.remove(fp_path)
            except OSError:
                pass
            return None
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)
    print(f"  Checkpoint carregado: {name}")
    return data

# --------------- Logging ------------------------------------
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("ptd_scraper")

print("Funções utilitárias carregadas.")

## 2. Scraping — Lista de Órgãos Signatários

Extrai os 91 órgãos e seus links de PDFs (Documento Diretivo + Anexo de Entregas) da página principal.

In [ ]:
# ============================================================
# CÉLULA 4 — Scraping: Lista de Órgãos e URLs dos PDFs
# ============================================================

def _classify_pdf_link(anchor_text: str) -> Optional[str]:
    """Classifica um link de PDF como 'diretivo' ou 'entregas' pelo texto da âncora."""
    text = normalize_text(anchor_text).lower()
    text_no_accents = strip_accents(text)

    diretivo_kw = ["diretivo", "documento diretivo"]
    if any(kw in text_no_accents for kw in diretivo_kw):
        return "diretivo"

    entregas_kw = ["entregas", "anexo de entregas", "anexo entregas"]
    if any(kw in text_no_accents for kw in entregas_kw):
        return "entregas"

    return None


def _extract_siglas_from_header(header_text: str) -> List[str]:
    """
    Extrai siglas de um cabeçalho como:
      'Plano de Transformação Digital SIGLA:'
      'Plano de Transformação Digital SIGLA1 / SIGLA2 / SIGLA3:'
      'Plano de Transformação Digital CVM -'
    """
    text = normalize_text(header_text)

    # Remove o prefixo "Plano de Transformação Digital"
    prefix_pattern = re.compile(
        r"Plano\s+de\s+Transforma[çc][ãa]o\s+Digital\s*", re.IGNORECASE
    )
    text = prefix_pattern.sub("", text).strip()

    # Remove trailing : ou -
    text = re.sub(r"[\s:–\-]+$", "", text).strip()

    if not text:
        return []

    # Divide por " / " para múltiplas siglas
    parts = [p.strip() for p in re.split(r"\s*/\s*", text) if p.strip()]

    # Filtra: siglas são uppercase (permitem hífen para SG-PR), 2-14 chars
    siglas = []
    for p in parts:
        # Limpa possíveis sufixos ("(NOVO)")
        p = re.sub(r"\s*\(.*?\)\s*", "", p).strip()
        if re.match(r"^[A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-ZÁÉÍÓÚÂÊÔÃÕÇ0-9\-]{1,14}$", p):
            siglas.append(p)

    return siglas


def scrape_organ_listing(url: str) -> List[OrganInfo]:
    """
    Faz o scraping da página gov.br para extrair órgãos e links de PDFs.

    Estrutura real da página:
      Cada órgão é um <td> contendo:
        <strong>Plano de Transformação Digital SIGLA:</strong>
        <a href="...diretivo.pdf">Documento Diretivo</a> /
        <a href="...entregas.pdf">Anexo de Entregas</a>
    """
    resp = safe_request(url)
    if resp is None:
        raise RuntimeError(f"Não foi possível acessar {url}")

    soup = BeautifulSoup(resp.content, "html.parser")

    # Encontrar todos os <strong> que contêm "Plano de Transformação Digital"
    organ_data: Dict[str, Dict[str, Optional[str]]] = {}
    seen_sigla_sets = set()  # evitar duplicatas

    for strong_tag in soup.find_all(["strong", "b"]):
        raw_text = strong_tag.get_text(separator=" ", strip=True)
        if "transformação digital" not in raw_text.lower() and "transformacao digital" not in raw_text.lower():
            continue

        siglas = _extract_siglas_from_header(raw_text)
        if not siglas:
            continue

        # Evitar processar duplicatas do mesmo grupo
        sigla_key = tuple(sorted(siglas))
        if sigla_key in seen_sigla_sets:
            continue
        seen_sigla_sets.add(sigla_key)

        # Encontrar links PDF no mesmo container (td, p, div, etc.)
        container = strong_tag.parent
        if container is None:
            continue

        pdf_links_in_container = []
        for a_tag in container.find_all("a", href=True):
            href = a_tag["href"]
            if not href.lower().endswith(".pdf"):
                continue
            if "ptds-vigentes/" not in href and "planos-de-transformacao-digital" not in href:
                continue
            anchor_text = a_tag.get_text(separator=" ", strip=True)
            doc_type = _classify_pdf_link(anchor_text)

            # Fallback: classificar pelo nome do arquivo
            if doc_type is None:
                fname = href.rsplit("/", 1)[-1].lower()
                if "diretivo" in fname or "diretiv" in fname:
                    doc_type = "diretivo"
                elif "entregas" in fname or "anexo" in fname:
                    doc_type = "entregas"
                else:
                    doc_type = "unknown"

            # Converter URL relativa para absoluta
            if href.startswith("/"):
                href = "https://www.gov.br" + href

            pdf_links_in_container.append((href, doc_type))

        # Atribuir URLs por tipo
        url_diretivo = None
        url_entregas = None
        for href, doc_type in pdf_links_in_container:
            if doc_type == "diretivo" and url_diretivo is None:
                url_diretivo = href
            elif doc_type == "entregas" and url_entregas is None:
                url_entregas = href
            elif doc_type == "unknown":
                if url_diretivo is None:
                    url_diretivo = href
                elif url_entregas is None:
                    url_entregas = href

        # Registrar para todas as siglas neste header
        for sigla in siglas:
            if sigla not in organ_data:
                organ_data[sigla] = {
                    "nome": raw_text,
                    "url_diretivo": url_diretivo,
                    "url_entregas": url_entregas,
                }

    logger.info(f"Scraping direto: {len(organ_data)} siglas encontradas")

    # Expandir grupos: membros herdam PDFs do cabeça se não tiverem próprios
    expanded = dict(organ_data)
    for head_sigla, members in ORGAN_GROUPS.items():
        if head_sigla in organ_data:
            head_info = organ_data[head_sigla]
            for member in members:
                if member == head_sigla:
                    continue
                if member not in expanded:
                    expanded[member] = {
                        "nome": head_info["nome"],
                        "url_diretivo": head_info["url_diretivo"],
                        "url_entregas": head_info["url_entregas"],
                    }
                else:
                    if expanded[member]["url_diretivo"] is None:
                        expanded[member]["url_diretivo"] = head_info["url_diretivo"]
                    if expanded[member]["url_entregas"] is None:
                        expanded[member]["url_entregas"] = head_info["url_entregas"]

    # Construir lista final
    organs: List[OrganInfo] = []
    for sigla in sorted(expanded.keys()):
        info = expanded[sigla]
        grupo = MEMBER_TO_GROUP.get(sigla)
        organs.append(OrganInfo(
            sigla=sigla,
            nome_completo=info["nome"],
            grupo=grupo,
            url_diretivo=info.get("url_diretivo"),
            url_entregas=info.get("url_entregas"),
        ))

    return organs


# ---- Execução (sempre faz scraping fresco — leva ~3s) ----
print("Fazendo scraping da página principal...")
all_organs = scrape_organ_listing(BASE_URL)

# ---- Validação e Resumo ----
_n_total = len(all_organs)
_n_diretivo = sum(1 for o in all_organs if o.url_diretivo)
_n_entregas = sum(1 for o in all_organs if o.url_entregas)
_n_ambos = sum(1 for o in all_organs if o.url_diretivo and o.url_entregas)
_n_nenhum = sum(1 for o in all_organs if not o.url_diretivo and not o.url_entregas)
_n_grupos = sum(1 for o in all_organs if o.grupo is not None)

print(f"\n{'='*50}")
print(f"Total de órgãos encontrados: {_n_total}")
if _n_total < 80 or _n_total > 110:
    print(f"  ⚠ ATENÇÃO: esperados ~91 órgãos, encontrados {_n_total}")
else:
    print(f"  ✓ Contagem dentro do esperado (~91)")
print(f"  Com Documento Diretivo:    {_n_diretivo}")
print(f"  Com Anexo de Entregas:     {_n_entregas}")
print(f"  Com ambos:                 {_n_ambos}")
print(f"  Sem nenhum PDF:            {_n_nenhum}")
print(f"  Membros de grupo:          {_n_grupos}")
print(f"{'='*50}")

if _n_nenhum > 0:
    print("\nÓrgãos SEM nenhum PDF:")
    for o in all_organs:
        if not o.url_diretivo and not o.url_entregas:
            print(f"  - {o.sigla}")

print("\nAmostra (primeiros 10):")
for o in all_organs[:10]:
    print(f"  {o.sigla:12s} | dir={'Sim' if o.url_diretivo else '---'} "
          f"| ent={'Sim' if o.url_entregas else '---'} "
          f"| grupo={o.grupo or '—'}")

## 3. Download dos PDFs

Baixa todos os PDFs identificados com rate-limiting e verificação de integridade.

In [ ]:
# ============================================================
# CÉLULA 5 — Download dos PDFs
# ============================================================

PDF_MAGIC_BYTES = b"%PDF"
MIN_SUSPICIOUS_SIZE = 10 * 1024   # 10 KB
MIN_VALID_SIZE = 1000             # 1 KB (skip threshold para resume)


def _download_single_pdf(url: str, dest_path: str, sigla: str,
                         doc_type: str) -> Optional[ProcessingError]:
    """
    Baixa um único PDF. Retorna ProcessingError em caso de falha, None se OK.
    Pula o download se o arquivo já existe com tamanho > MIN_VALID_SIZE.
    """
    # Resume: pula se já existe e parece válido
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > MIN_VALID_SIZE:
        return None

    resp = safe_request(url)
    if resp is None:
        return ProcessingError(
            orgao_sigla=sigla,
            document_type=doc_type,
            stage="download",
            error_type="request_failed",
            error_message=f"Falha ao baixar após {MAX_RETRIES} tentativas",
            url=url,
        )

    content = resp.content

    # Verifica magic bytes
    if not content[:4].startswith(PDF_MAGIC_BYTES):
        return ProcessingError(
            orgao_sigla=sigla,
            document_type=doc_type,
            stage="download",
            error_type="invalid_pdf",
            error_message=f"Arquivo não começa com %PDF (magic bytes: {content[:8]!r})",
            url=url,
        )

    # Salva o arquivo
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    with open(dest_path, "wb") as f:
        f.write(content)

    # Alerta para arquivos suspeitamente pequenos
    file_size = len(content)
    if file_size < MIN_SUSPICIOUS_SIZE:
        logger.warning(f"  {sigla}/{doc_type}: arquivo pequeno ({file_size:,} bytes) — pode estar corrompido")

    return None


def download_all_pdfs(organs: List[OrganInfo]) -> List[ProcessingError]:
    """
    Baixa todos os PDFs (diretivo + entregas) dos órgãos.
    Atualiza organ.pdf_path_diretivo e organ.pdf_path_entregas.
    Retorna lista de erros.
    """
    errors: List[ProcessingError] = []
    downloaded_urls: Dict[str, str] = {}   # url → caminho local (evita re-download)

    # Contagem total de downloads potenciais
    total_downloads = sum(
        (1 if o.url_diretivo else 0) + (1 if o.url_entregas else 0)
        for o in organs
    )

    pbar = tqdm(total=total_downloads, desc="Baixando PDFs", unit="pdf")

    for organ in organs:
        for doc_type, url_attr, path_attr, target_dir in [
            ("diretivo", "url_diretivo", "pdf_path_diretivo", DIRS["pdfs_diretivo"]),
            ("entregas", "url_entregas", "pdf_path_entregas", DIRS["pdfs_entregas"]),
        ]:
            url = getattr(organ, url_attr)
            if url is None:
                continue

            filename = f"{organ.sigla}_{doc_type}.pdf"
            dest_path = os.path.join(target_dir, filename)

            # Se outro órgão do mesmo grupo já baixou este URL, reutiliza
            if url in downloaded_urls:
                existing_path = downloaded_urls[url]
                if os.path.exists(existing_path):
                    # Copia ou cria link — usamos cópia para robustez
                    if not os.path.exists(dest_path):
                        import shutil
                        shutil.copy2(existing_path, dest_path)
                    setattr(organ, path_attr, dest_path)
                    pbar.update(1)
                    continue

            err = _download_single_pdf(url, dest_path, organ.sigla, doc_type)
            if err is not None:
                errors.append(err)
            else:
                setattr(organ, path_attr, dest_path)
                downloaded_urls[url] = dest_path

            pbar.update(1)

    pbar.close()
    return errors


# ---- Execução ----
# Download já faz skip automático de PDFs existentes (resume embutido no _download_single_pdf)
if all_organs:
    print(f"Iniciando download de PDFs para {len(all_organs)} órgãos...")
    print("(PDFs já baixados serão reutilizados automaticamente)")
    download_errors = download_all_pdfs(all_organs)
    all_errors.extend(download_errors)
else:
    print("ERRO: Nenhum órgão encontrado. Verifique a célula de scraping.")
    download_errors = []

# ---- Resumo ----
_n_dir_ok = sum(1 for o in all_organs if o.pdf_path_diretivo and os.path.exists(o.pdf_path_diretivo))
_n_ent_ok = sum(1 for o in all_organs if o.pdf_path_entregas and os.path.exists(o.pdf_path_entregas))

_total_size = 0
_suspicious: List[str] = []
for o in all_organs:
    for p in [o.pdf_path_diretivo, o.pdf_path_entregas]:
        if p and os.path.exists(p):
            sz = os.path.getsize(p)
            _total_size += sz
            if sz < MIN_SUSPICIOUS_SIZE:
                _suspicious.append(f"{o.sigla}: {os.path.basename(p)} ({sz:,} bytes)")

print(f"\n{'='*50}")
print(f"Download concluído")
print(f"  Documento Diretivo OK:    {_n_dir_ok}")
print(f"  Anexo de Entregas OK:     {_n_ent_ok}")
print(f"  Erros de download:        {len(download_errors)}")
print(f"  Tamanho total:            {_total_size / (1024*1024):.1f} MB")
print(f"{'='*50}")

if _suspicious:
    print(f"\nArquivos suspeitamente pequenos (<{MIN_SUSPICIOUS_SIZE//1024} KB):")
    for s in _suspicious:
        print(f"  - {s}")

if download_errors:
    print(f"\nErros de download:")
    for e in download_errors:
        print(f"  - {e.orgao_sigla}/{e.document_type}: {e.error_type} — {e.error_message}")

In [ ]:
# ============================================================
# CÉLULA 5b — Dedup por hash MD5 (PDFs compartilhados)
# ============================================================
# Órgãos que pertencem a um grupo ministerial (ORGAN_GROUPS) frequentemente
# compartilham um único PDF publicado pelo ministério. O scraping baixa uma
# cópia para cada sigla membro, mas o conteúdo é idêntico. Sem dedup, a
# extração processaria o mesmo conteúdo N vezes, gerando duplicatas.
#
# Estratégia: para cada hash MD5 distinto, manter como "owner" a sigla
# alfabeticamente menor. As demais ficam com path=None — a extração as
# ignora. A cobertura é expandida via MEMBER_TO_GROUP, marcando os
# membros não-owner como "compartilhado".
import hashlib
from collections import defaultdict


def _md5(path: str) -> Optional[str]:
    if not path or not os.path.exists(path):
        return None
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def dedupe_by_hash(organs: List[OrganInfo], path_attr: str) -> int:
    """Para cada hash duplicado, zera path em todos exceto o sigla
    alfabético menor. Retorna número de cópias zeradas."""
    by_hash: Dict[str, List[OrganInfo]] = defaultdict(list)
    for o in organs:
        p = getattr(o, path_attr, None)
        h = _md5(p)
        if h:
            by_hash[h].append(o)
    n_dropped = 0
    for h, members in by_hash.items():
        if len(members) <= 1:
            continue
        members.sort(key=lambda o: o.sigla)
        for o in members[1:]:
            setattr(o, path_attr, None)
            n_dropped += 1
    return n_dropped


_n_dir = dedupe_by_hash(all_organs, "pdf_path_diretivo")
_n_ent = dedupe_by_hash(all_organs, "pdf_path_entregas")

print(f"\n{'='*50}")
print(f"Dedup por hash MD5")
print(f"  PDFs diretivo descartados (cópias):  {_n_dir}")
print(f"  PDFs entregas descartados (cópias):  {_n_ent}")
print(f"  Owners únicos diretivo: {sum(1 for o in all_organs if o.pdf_path_diretivo)}")
print(f"  Owners únicos entregas: {sum(1 for o in all_organs if o.pdf_path_entregas)}")
print(f"{'='*50}")

In [ ]:
# ============================================================
# CÉLULA 5d — Metadados dos PDFs (datas, tamanho)
# ============================================================
# Após o dedup (5c), cada PDF único sobrevive sob seu "owner" (a sigla
# alfabeticamente menor; os não-owners ficam com pdf_path=None). Esta célula
# registra, por PDF presente no disco, as datas de criação/modificação lidas do
# próprio arquivo (via PyMuPDF) e o tamanho em KB -> output/pdf_metadata.csv.
#
# Consumido por build_manifest.py (contagens de PDF) e exigido pelo bundle de
# publicação (13c). Uma linha por PDF único (tantas quantas URLs distintas).
import fitz


def _parse_pdf_date(raw: str) -> str:
    """Data de PDF (ex.: "D:20260429120000-03'00'") -> "YYYY-MM-DD". "" se ausente."""
    if not raw:
        return ""
    s = raw[2:] if raw.startswith("D:") else raw
    return f"{s[0:4]}-{s[4:6]}-{s[6:8]}" if len(s) >= 8 and s[:8].isdigit() else ""


def _pdf_file_meta(path: str) -> dict:
    """Datas (lidas do PDF) + tamanho_kb (do disco). Datas vazias se ilegíveis."""
    cre = mod = ""
    try:
        with fitz.open(path) as doc:
            meta = doc.metadata or {}
        cre = _parse_pdf_date(meta.get("creationDate", ""))
        mod = _parse_pdf_date(meta.get("modDate", ""))
    except Exception:
        pass   # PDF sem metadados / corrompido: datas vazias, tamanho ainda conta
    return {"data_criacao_pdf": cre, "data_modificacao_pdf": mod,
            "tamanho_kb": round(os.path.getsize(path) / 1024)}


def build_pdf_metadata(organs: "List[OrganInfo]") -> "List[dict]":
    """Uma linha por PDF presente no disco (owner pós-dedup), ord. por (sigla, tipo)."""
    rows = []
    for o in organs:
        for tipo, path in (("diretivo", o.pdf_path_diretivo),
                           ("entregas", o.pdf_path_entregas)):
            if path and os.path.exists(path):
                rows.append({"sigla": o.sigla, "tipo": tipo,
                             **_pdf_file_meta(path), "vigencia": ""})
    rows.sort(key=lambda r: (r["sigla"], r["tipo"]))
    return rows


# ---- Execução ----
PDF_METADATA_COLS = ["sigla", "tipo", "data_criacao_pdf", "data_modificacao_pdf",
                     "vigencia", "tamanho_kb"]
if all_organs:
    _pdf_meta_rows = build_pdf_metadata(all_organs)
    _pm_path = os.path.join(DIRS["output"], "pdf_metadata.csv")
    pd.DataFrame(_pdf_meta_rows, columns=PDF_METADATA_COLS).to_csv(
        _pm_path, index=False, encoding="utf-8-sig")
    print(f"pdf_metadata.csv gravado — {len(_pdf_meta_rows)} PDFs únicos.")
else:
    print("Nenhum órgão — pdf_metadata.csv não gerado.")

## 4. Configuração do Extrator e Classificação de Tabelas

Inicializa o extrator de tabelas (PyMuPDF `find_tables()`) e define funções de classificação para tabelas de riscos e entregas.

In [ ]:
# ============================================================
# CÉLULA 6 — Configuração do Extrator de Tabelas (PyMuPDF)
# ============================================================

import fitz  # PyMuPDF

# --------------- Classificação de tabelas diretivo ------------

def _normalize_header(text: str) -> str:
    if not isinstance(text, str): return ""
    t = normalize_text(text).lower()
    t = t.replace("- ", "").replace("-\n", "")
    return strip_accents(t)


def classify_diretivo_table(df: pd.DataFrame) -> str:
    if df is None or len(df.columns) == 0: return "unknown"
    ncols = len(df.columns)
    headers = [_normalize_header(str(c)) for c in df.columns]
    first_row = [_normalize_header(str(v)) for v in df.iloc[0]] if len(df) > 0 else []
    combined = " ".join(headers + first_row)

    risk_kw = ["risco", "probabilidade", "impacto", "tratamento", "ocorrer"]
    risk_alt = ["evento", "classificacao", "severidade", "resposta", "acao", "acoes",
                "id do risco", "descricao do risco", "opcao de tratamento"]
    risk_hits = sum(1 for kw in risk_kw if kw in combined)
    risk_alt_hits = sum(1 for kw in risk_alt if kw in combined)

    if risk_hits >= 2 and 3 <= ncols <= 8: return "risk_table"
    if risk_hits >= 1 and risk_alt_hits >= 1 and 3 <= ncols <= 8: return "risk_table"
    if "risco" in combined and "tratamento" in combined and 3 <= ncols <= 8: return "risk_table"

    info_kw = ["orgao", "ministerio", "secretaria", "sigla", "cnpj", "responsavel",
               "gestor", "dirigente", "titular", "instituicao", "vinculacao"]
    if sum(1 for kw in info_kw if kw in combined) >= 2: return "organ_info"

    sig_kw = ["assinatura", "assinado", "data", "nome", "cargo", "cpf"]
    if sum(1 for kw in sig_kw if kw in combined) >= 2 and ncols <= 4: return "signature"

    return "unknown"


def classify_entregas_table(df: pd.DataFrame) -> str:
    if df is None or df.empty: return "unknown"
    headers = [_normalize_header(str(c)) for c in df.columns]
    first_row = [_normalize_header(str(v)) for v in df.iloc[0]] if len(df) > 0 else []
    combined = " ".join(headers + first_row)
    compact = combined.replace(" ", "")

    if "justificativa" in combined: return "canceladas"
    if "dtentrega" in compact or "data entrega" in combined or "data de entrega" in combined: return "concluidas"
    if "pactuado?" in combined or "pactuado ?" in combined: return "concluidas"
    if ("area" in combined and "responsavel" in combined) and "dtpactuada" in compact: return "pactuadas"
    if "dtpactuada" in compact or "data pactuada" in combined: return "pactuadas"
    return "unknown"


def _cols_are_data(df: pd.DataFrame) -> bool:
    """Detecta se find_tables interpretou o 1o risco como header de coluna."""
    if df.shape[1] < 4: return False
    col0 = _normalize_header(str(df.columns[0]))
    if len(col0) < 10 or col0.startswith("risco") or col0.startswith("col") or col0 == "nan":
        return False
    col1 = _normalize_header(str(df.columns[1]))
    scale_vals = ["raro", "pouco provavel", "provavel", "muito provavel", "praticamente certo",
                  "baixo", "medio", "alto", "muito alto", "baixa", "media", "alta"]
    return any(sv in col1 for sv in scale_vals)


def _is_risk_data(df: pd.DataFrame) -> bool:
    """Verifica se uma tabela contém valores de escala de risco (continuação)."""
    if df is None or df.empty or df.shape[1] < 4: return False
    all_text = strip_accents(" ".join(str(v).lower() for v in df.values.flatten()))
    scale_vals = ["raro", "pouco provavel", "provavel", "muito provavel", "praticamente certo",
                  "muito baixo", "baixo", "medio", "alto", "muito alto",
                  "mitigar", "eliminar", "transferir", "aceitar", "baixa", "media", "alta"]
    return sum(1 for sv in scale_vals if sv in all_text) >= 2


def _is_subheader_row(row) -> bool:
    text = strip_accents(" ".join(str(v) for v in row).lower())
    return any(kw in text for kw in ["certo]", "certo ]", "ocorrer", "muito alto]",
               "muito alto ]", "tratamento do risco", "escolher entre"])


def _consolidate_multiline_cells(df: pd.DataFrame) -> pd.DataFrame:
    """Consolida tabelas onde cada risco ocupa múltiplas linhas porque o
    PDF usa células com texto quebrado em várias linhas internas.

    Heurística: se col0 (ID do risco) está populado em <40% das linhas e
    há ao menos 3 IDs distintos, agrupa as linhas entre IDs concatenando
    os textos de cada coluna.
    """
    if df is None or df.empty or df.shape[1] < 4:
        return df

    n = len(df)

    def _is_id(v):
        s = str(v).strip()
        return bool(s) and s.lower() not in ("nan", "none")

    id_idx = [i for i in range(n) if _is_id(df.iloc[i, 0])]
    if len(id_idx) < 3 or len(id_idx) >= 0.4 * n:
        return df

    # Cria blocos: [start : next_id_row]. start é a linha do ID exceto
    # quando a linha imediatamente anterior contém texto em col1 sem ID
    # (caso comum em PDFs onde a primeira linha do risco aparece visualmente
    # acima do ID quando há quebra de página/coluna).
    def _has_text(v):
        s = str(v).strip()
        return bool(s) and s.lower() not in ("nan", "none")

    blocks = []
    prev_end = 0
    for k, id_pos in enumerate(id_idx):
        start = id_pos
        if (start > prev_end and not _is_id(df.iloc[start - 1, 0])
                and _has_text(df.iloc[start - 1, 1])):
            start -= 1
        end = id_idx[k + 1] if k + 1 < len(id_idx) else n
        blocks.append((start, end))
        prev_end = end

    consolidated_rows = []
    for start, end in blocks:
        merged = []
        for ci in range(df.shape[1]):
            parts = []
            for ri in range(start, end):
                v = str(df.iloc[ri, ci]).strip()
                if v and v.lower() not in ("nan", "none"):
                    parts.append(v)
            # Preserva semântica de bullets: se algum chunk começa com "-"
            # ou contém ";", junta com " | " (mais legível downstream).
            # Caso simples (texto contínuo): mantém join com espaço.
            has_bullets = any(p.lstrip().startswith(("-", "•", "*")) for p in parts)
            has_semicolons = any(";" in p for p in parts) and len(parts) > 1
            sep = " | " if (has_bullets or has_semicolons) else " "
            merged.append(sep.join(parts))
        consolidated_rows.append(merged)

    return pd.DataFrame(consolidated_rows, columns=df.columns)


def _is_orphan_risk_data(df: pd.DataFrame) -> bool:
    """Detecta tabela cujo conteúdo é dado de risco mas find_tables não
    associou a um cabeçalho identificável (retornou Col0/Col1/... ou
    fragmentos do template como headers). Permite processar tabelas que
    não foram precedidas por uma tabela com cabeçalho válido (caso em
    que is_continuation não dispara por falta de risk_ncols)."""
    if df is None or df.empty or df.shape[1] < 4 or df.shape[1] > 8:
        return False
    if not _is_risk_data(df):
        return False
    headers_norm = [_normalize_header(str(c)) for c in df.columns]
    n_generic = sum(1 for h in headers_norm
                    if (h.startswith("col") and len(h) <= 5)
                    or any(kw in h for kw in ["escolher entre", "certo]", "certo ]",
                                              "muito alto]", "muito alto ]", "ocorrer"]))
    return n_generic >= max(2, len(headers_norm) // 2)


def _extract_action_list(doc_text: str) -> dict:
    """Extrai lista 'Referencial para ações de tratamento do risco'."""
    actions = {}
    for pat in [r"[Rr]eferencial\s+para\s+a[çc][õo]es\s+de\s+tratamento",
                r"[Aa][çc][õo]es\s+de\s+tratamento\s+do\s+risco\s*:"]:
        m = re.search(pat, doc_text)
        if m:
            for line in doc_text[m.end():].split('\n'):
                line = line.strip()
                am = re.match(r"^(\d{1,2})\s*[\.\-\)]\s*(.+)", line)
                if am:
                    actions[am.group(1)] = am.group(2).strip()
                elif actions and not line[0:1].isdigit() and len(actions) > 3:
                    break
            if actions: return actions
    return actions


def _resolve_action_refs(acoes_text: str, action_list: dict) -> str:
    """Resolve referências numéricas ('1, 2, 9') para texto completo."""
    if not acoes_text or not action_list: return acoes_text
    refs = re.findall(r'\d+', acoes_text)
    if not refs: return acoes_text
    tokens = re.split(r'[,;\s]+', acoes_text.strip())
    num_tokens = sum(1 for t in tokens if re.match(r'^\d+$', t.strip()))
    if num_tokens / max(len(tokens), 1) < 0.5: return acoes_text
    resolved = [f"{ref}. {action_list[ref]}" for ref in refs if ref in action_list]
    return " | ".join(resolved) if resolved else acoes_text


print("PyMuPDF configurado.")
print("Classificadores de tabelas e funções de extração carregados.")
print(f"Produtos no vocabulário: {len(ALL_PRODUTOS)}")

## 5. Extração de Tabelas de Risco (Documentos Diretivos)

Extrai e normaliza a tabela de riscos de cada Documento Diretivo. Inclui merge multi-página, recuperação de header-as-data e resolução de ações numéricas.

In [ ]:
# ============================================================
# CÉLULA 7 — Extração de Tabelas de Risco (PyMuPDF find_tables)
# ============================================================
# Inclui: merge multi-página, recuperação header-as-data,
# resolução de referências numéricas de ações.

_HEADER_LITERALS_RISK = {
    "risco", "probabilidade", "impacto", "tratamento",
    "acoes", "ações", "ação",
    "probabilidade de ocorrencia", "probabilidade de ocorrência",
    "nivel de impacto", "nível de impacto",
    "id risco", "id do risco", "id",
}


def _is_header_literal(value: str) -> bool:
    """True se valor é texto literal de header (capturado como dado)."""
    if not value:
        return False
    norm = strip_accents(normalize_text(value).lower().strip())
    return norm in _HEADER_LITERALS_RISK


def _try_swap_prob_impacto(prob_raw: str, imp_raw: str,
                           prob_m: Tuple[str, float],
                           imp_m: Tuple[str, float]) -> Optional[Tuple[str, str, Tuple[str, float], Tuple[str, float]]]:
    """Detecta column-shift prob↔impacto e retorna valores corrigidos.

    Aciona se ambos casam mal nas escalas esperadas E casam bem se invertidos.
    Conservador: requer score >= 0.85 nas escalas invertidas.

    Retorna (prob_corrigido, imp_corrigido, prob_m_corrigido, imp_m_corrigido)
    ou None se não houver evidência clara de shift.
    """
    if not prob_raw or not imp_raw:
        return None
    if prob_m[1] >= 0.70 and imp_m[1] >= 0.70:
        return None  # ambos casam onde esperado — não precisa swap
    alt_prob_in_imp = fuzzy_match_scale(prob_raw, IMPACTO_SCALE)
    alt_imp_in_prob = fuzzy_match_scale(imp_raw, PROBABILIDADE_SCALE)
    if alt_prob_in_imp[1] >= 0.85 and alt_imp_in_prob[1] >= 0.85:
        # Swap evidente: prob_raw é valor de impacto, imp_raw é valor de prob
        return (imp_raw, prob_raw, alt_imp_in_prob, alt_prob_in_imp)
    return None


def _map_risk_columns(df: pd.DataFrame) -> Dict[str, Optional[str]]:
    canonical = {"risco":None,"probabilidade":None,"impacto":None,"tratamento":None,"acoes":None}
    keyword_map = {
        "risco": ["risco","evento","descricao do risco","descricao"],
        "probabilidade": ["probabilidade","probabilidade de ocorrer","prob","classificacao de probabilidade"],
        "impacto": ["impacto","severidade","classificacao de impacto"],
        "tratamento": ["opcao de tratamento","tratamento","resposta","tipo de tratamento","estrategia"],
        "acoes": ["acoes de tratamento","descrever acoes","acoes","acao","medidas","plano de acao"],
    }
    headers = {str(c): _normalize_header(str(c)) for c in df.columns}
    cols_list = [str(c) for c in df.columns]

    # ID column ("ID do risco", "Nº", etc) deve ser excluído do match para "risco"
    id_cols = {c for c, n in headers.items() if n in (
        "id do risco", "id risco", "id", "n", "no", "num", "numero", "codigo", "cod"
    ) or n.startswith("id ")}

    for canon_key, keywords in keyword_map.items():
        best_col, best_score = None, 0.0
        for col_name, col_norm in headers.items():
            if canon_key == "risco" and col_name in id_cols:
                continue
            for kw in keywords:
                if kw in col_norm:
                    score = max(len(kw)/max(len(col_norm),1), 0.85)
                    if score > best_score: best_score, best_col = score, col_name
            if best_col is None:
                for kw in keywords:
                    ratio = difflib.SequenceMatcher(None, col_norm, strip_accents(kw)).ratio()
                    if ratio > best_score and ratio >= 0.65: best_score, best_col = ratio, col_name
        canonical[canon_key] = best_col

    # Fallback posicional: quando o cabeçalho do PDF é genérico (Col0/Col2)
    # ou fragmentado, usa a posição padrão do template SGD (5 colunas).
    # Aplica somente se a coluna posicional estiver livre — não sobrescreve
    # mapeamentos por keyword.
    fallback_pos = {"risco": 0, "probabilidade": 1, "impacto": 2, "tratamento": 3, "acoes": 4}
    used_cols = {v for v in canonical.values() if v is not None}
    # Detecta offset por id_risco quando ncols >= 6 e a primeira coluna parece ID
    has_id_col = bool(id_cols) and cols_list[0] in id_cols
    offset = 1 if (len(cols_list) >= 6 and (
        has_id_col or _normalize_header(cols_list[0]) == "col0"
    )) else 0
    for field, pos in fallback_pos.items():
        if canonical[field] is None and pos + offset < len(cols_list):
            cand = cols_list[pos + offset]
            if cand not in used_cols and cand not in id_cols:
                canonical[field] = cand
                used_cols.add(cand)
    return canonical


def extract_risk_table(pdf_path: str, sigla: str) -> Tuple[List[RiskEntry], List[ProcessingError]]:
    entries: List[RiskEntry] = []
    errors: List[ProcessingError] = []

    try:
        doc = fitz.open(pdf_path)
    except Exception as exc:
        errors.append(ProcessingError(orgao_sigla=sigla, document_type="diretivo",
            stage="extraction", error_type="pdf_open_failed", error_message=str(exc)))
        return entries, errors

    # Extrair lista de ações de tratamento
    full_text = "\n".join(p.get_text() for p in doc)
    action_list = _extract_action_list(full_text)

    risk_ncols = None
    col_order = None

    for page in doc:
        tabs = page.find_tables()
        if not tabs.tables:
            continue
        for table in tabs.tables:
            try:
                df = table.to_pandas()
            except Exception:
                continue
            if df is None or df.shape[1] < 4:
                continue

            df = _consolidate_multiline_cells(df)

            has_header = classify_diretivo_table(df) == "risk_table"
            data_as_header = _cols_are_data(df)
            is_continuation = (risk_ncols and df.shape[1] == risk_ncols
                               and not has_header and _is_risk_data(df))
            is_orphan = (not has_header and not data_as_header and not is_continuation
                         and _is_orphan_risk_data(df))

            if not has_header and not data_as_header and not is_continuation and not is_orphan:
                continue

            if has_header:
                col_map = _map_risk_columns(df)
                if col_map["risco"] is None and len(df.columns) > 0:
                    col_map["risco"] = str(df.columns[0])
                col_order = list(col_map.keys())
                risk_ncols = len(df.columns)

                if len(df) > 0 and _is_subheader_row(df.iloc[0]):
                    df = df.iloc[1:].reset_index(drop=True)
                if len(df) == 0:
                    continue

            elif data_as_header:
                risk_ncols = len(df.columns)
                if col_order is None:
                    col_order = ["risco", "probabilidade", "impacto", "tratamento", "acoes"]
                    if df.shape[1] >= 6:
                        col_order = ["id_risco"] + col_order

                # Recuperar primeiro risco do header
                vals = [normalize_text(str(c)) for c in df.columns]
                vals = ["" if v.lower() in ("nan", "none") else v for v in vals]
                if vals[0] and not _is_subheader_row(vals):
                    prob_raw = vals[1] if len(vals) > 1 else ""
                    imp_raw = vals[2] if len(vals) > 2 else ""
                    trat_raw = vals[3] if len(vals) > 3 else ""
                    acoes_raw = vals[4] if len(vals) > 4 else ""
                    prob_m = fuzzy_match_scale(prob_raw, PROBABILIDADE_SCALE)
                    imp_m = fuzzy_match_scale(imp_raw, IMPACTO_SCALE)
                    trat_m = fuzzy_match_scale(trat_raw, TRATAMENTO_OPTIONS)
                    entries.append(RiskEntry(
                        orgao_sigla=sigla, risco_texto=vals[0],
                        probabilidade_original=prob_raw,
                        probabilidade_normalizada=prob_m[0] if prob_m[1] >= 0.70 else "",
                        impacto_original=imp_raw,
                        impacto_normalizado=imp_m[0] if imp_m[1] >= 0.70 else "",
                        tratamento_original=trat_raw,
                        tratamento_normalizado=trat_m[0] if trat_m[1] >= 0.70 else "",
                        acoes_tratamento=_resolve_action_refs(acoes_raw, action_list),
                        extraction_confidence="medium",
                        needs_review=True,
                        review_reason="recuperado de header de coluna",
                    ))

            elif is_continuation and col_order:
                if len(df) > 0 and _is_subheader_row(df.iloc[0]):
                    df = df.iloc[1:].reset_index(drop=True)

            elif is_orphan:
                risk_ncols = len(df.columns)
                if col_order is None:
                    col_order = ["risco", "probabilidade", "impacto", "tratamento", "acoes"]
                    if df.shape[1] >= 6:
                        col_order = ["id_risco"] + col_order
                if len(df) > 0 and _is_subheader_row(df.iloc[0]):
                    df = df.iloc[1:].reset_index(drop=True)
                if len(df) == 0:
                    continue

            if col_order is None:
                continue

            mapped_count = sum(1 for k in col_order if k in (col_map if has_header else {}))
            if has_header:
                active_map = col_map
            else:
                # Mapeamento posicional. Detecta offset de id_risco quando a
                # tabela tem 1 coluna a mais que o col_order esperado e a 1ª
                # coluna parece um identificador curto (single char/num).
                cols_list = [str(c) for c in df.columns]
                offset = 0
                if len(cols_list) > len(col_order) and len(df) > 0:
                    first_vals = [str(df.iloc[r, 0]).strip() for r in range(min(3, len(df)))]
                    if all(len(v) <= 3 and v.lower() not in ("nan","none","") for v in first_vals if v):
                        offset = 1
                active_map = {f: cols_list[i + offset]
                              for i, f in enumerate(col_order)
                              if i + offset < len(cols_list)}

            for _, row in df.iterrows():
                if _is_subheader_row(row):
                    continue

                def _get(field):
                    col = active_map.get(field)
                    if col and col in row.index:
                        v = normalize_text(str(row[col]))
                        return "" if v.lower() in ("nan", "none") else v
                    return ""

                risco = _get("risco")
                prob = _get("probabilidade")
                imp = _get("impacto")
                trat = _get("tratamento")
                acoes = _get("acoes")

                if not risco and not prob and not imp:
                    continue

                # FIX ESTRUTURAL 1 — Header capturado como dado: skip row inteira.
                # Acontece em PDFs onde find_tables pega header repetido em
                # página seguinte como primeira linha de dados (CENSIPAM, MJSP).
                if (_is_header_literal(risco) or _is_header_literal(prob)
                        or _is_header_literal(imp) or _is_header_literal(trat)):
                    continue

                prob_m = fuzzy_match_scale(prob, PROBABILIDADE_SCALE)
                imp_m = fuzzy_match_scale(imp, IMPACTO_SCALE)

                # FIX ESTRUTURAL 2 — Swap automático prob↔impacto.
                # Detecta column-shift simétrico (CADE: "Médio"/"1-Alto" em
                # prob, valor de prob em impacto). Conservador: só swap se
                # ambos casam >=0.85 nas escalas INVERTIDAS.
                swap_result = _try_swap_prob_impacto(prob, imp, prob_m, imp_m)
                swap_applied = False
                if swap_result:
                    prob, imp, prob_m, imp_m = swap_result
                    swap_applied = True

                trat_m = fuzzy_match_scale(trat, TRATAMENTO_OPTIONS)
                acoes_resolved = _resolve_action_refs(acoes, action_list)

                review_reasons = []
                if swap_applied:
                    review_reasons.append("swap automático prob↔impacto (column-shift corrigido)")
                if prob and prob_m[1] < 0.70: review_reasons.append(f"probabilidade: '{prob[:40]}'")
                if imp and imp_m[1] < 0.70: review_reasons.append(f"impacto: '{imp[:40]}'")
                if not risco: review_reasons.append("risco vazio")
                # FIX ESTRUTURAL 3 — Column bleed em impacto/tratamento.
                # Sinaliza vazamento sem corrigir (não dá pra recuperar dado
                # original sem reprocessar PDF). Threshold: >100 chars E sem
                # match canônico.
                if imp and len(imp) > 100 and imp_m[1] < 0.70:
                    review_reasons.append(f"impacto: bleed de coluna ({len(imp)} chars)")
                if trat and len(trat) > 100 and trat_m[1] < 0.70:
                    review_reasons.append(f"tratamento: bleed de coluna ({len(trat)} chars)")

                confidence = ("high" if not review_reasons else
                              "medium" if len(review_reasons) <= 1 else "low")

                entries.append(RiskEntry(
                    orgao_sigla=sigla, risco_texto=risco,
                    probabilidade_original=prob,
                    probabilidade_normalizada=prob_m[0] if prob_m[1] >= 0.70 else "",
                    impacto_original=imp,
                    impacto_normalizado=imp_m[0] if imp_m[1] >= 0.70 else "",
                    tratamento_original=trat,
                    tratamento_normalizado=trat_m[0] if trat_m[1] >= 0.70 else "",
                    acoes_tratamento=acoes_resolved,
                    extraction_confidence=confidence,
                    needs_review=len(review_reasons) > 0,
                    review_reason="; ".join(review_reasons) if review_reasons else None,
                ))

    doc.close()
    if not entries:
        errors.append(ProcessingError(orgao_sigla=sigla, document_type="diretivo",
            stage="extraction", error_type="no_risk_table",
            error_message=f"Nenhuma tabela de risco encontrada em {os.path.basename(pdf_path)}"))

    return entries, errors


# --------------- Extração em lote -----------------------------

def extract_all_risks() -> None:
    global all_risks, all_errors

    # Fingerprint do estado upstream (lista de siglas com PDF diretivo após dedup).
    # Se 05c_dedup.py rodar e zerar paths, o fingerprint muda e o cache é
    # invalidado automaticamente, evitando regressões silenciosas.
    fp = state_fingerprint(sorted((o.sigla, bool(o.pdf_path_diretivo)) for o in all_organs))

    cached = load_checkpoint("risks_raw", expected_fingerprint=fp)
    if cached is not None and len(cached[0]) > 0:
        cached_risks, cached_errors, processed_siglas = cached
        all_risks.extend(cached_risks)
        all_errors.extend(cached_errors)
        print(f"  Retomando: {len(cached_risks)} riscos de {len(processed_siglas)} órgãos")
    else:
        cached_risks, cached_errors, processed_siglas = [], [], set()

    organs_with_pdf = [o for o in all_organs if o.pdf_path_diretivo]
    pending = [o for o in organs_with_pdf if o.sigla not in processed_siglas]

    if not pending:
        print("  Todos os órgãos já processados (checkpoint).")
        return

    print(f"  Processando: {len(pending)} órgãos pendentes")

    pdf_results_cache: Dict[str, Tuple[List[RiskEntry], List[ProcessingError], str]] = {}
    batch_risks, batch_errors = [], []
    count = 0

    for organ in tqdm(pending, desc="Extraindo riscos"):
        sigla = organ.sigla
        pdf_path = organ.pdf_path_diretivo

        if not os.path.isfile(pdf_path):
            batch_errors.append(ProcessingError(orgao_sigla=sigla, document_type="diretivo",
                stage="extraction", error_type="file_not_found",
                error_message=f"PDF não encontrado: {pdf_path}"))
            processed_siglas.add(sigla)
            count += 1
            continue

        real_path = os.path.realpath(pdf_path)
        if real_path in pdf_results_cache:
            owner = pdf_results_cache[real_path][2]
            processed_siglas.add(sigla)
            logger.info(f"[{sigla}] PDF compartilhado com {owner} — sem duplicação")
        else:
            entries, errs = extract_risk_table(pdf_path, sigla)
            pdf_results_cache[real_path] = (entries, errs, sigla)
            batch_risks.extend(entries)
            all_risks.extend(entries)
            batch_errors.extend(errs)
            all_errors.extend(errs)
            processed_siglas.add(sigla)
            if entries:
                logger.info(f"[{sigla}] {len(entries)} riscos extraídos")

        count += 1
        if count % 10 == 0:
            save_checkpoint((cached_risks + batch_risks, cached_errors + batch_errors, processed_siglas), "risks_raw", fingerprint=fp)

    save_checkpoint((cached_risks + batch_risks, cached_errors + batch_errors, processed_siglas), "risks_raw", fingerprint=fp)
    print(f"  Extração de riscos concluída.")


# --------------- Execução -------------------------------------
extract_all_risks()


# --------------- Auditoria pós-extração (Categoria A) ---------
# Detecta padrões anômalos comuns na extração tabular e marca needs_review
# com motivo explícito. Não descarta entry — preserva audit trail.
#
# Padrões detectados:
#   (a) Header capturado como dado (valor == nome literal do campo)
#   (b) Fragmento de hifenização (texto < 5 chars OU começa/termina com "-"
#       sem letra ao lado)
#   (c) Bleed de coluna em campo de escala (texto > 30 chars em
#       prob/imp/trat, que deveria ser categoria curta)
#   (d) Column-shift detectável (valor canoniza melhor em outra escala)

_HEADER_LITERALS = {
    "probabilidade", "impacto", "tratamento", "risco",
    "probabilidade de ocorrencia", "probabilidade de ocorrência",
    "nivel de impacto", "nível de impacto",
}
_SCALE_MAX_LEN = 30  # categorias de escala raramente passam de 25 chars


def _is_header_capture(value: str) -> bool:
    """Valor literal igual ao nome do campo."""
    if not value:
        return False
    norm = strip_accents(normalize_text(value).lower().strip())
    return norm in _HEADER_LITERALS


def _is_fragment(value: str) -> bool:
    """Texto fragmentado por hifenização ou quebra de página.

    Detecção CONSERVADORA — só pega padrões inequívocos pra evitar
    falsos positivos em valores válidos curtos ("raro", "alto", "3").

    Padrões:
    1. Termina com hífen sem palavra-seguinte (resto da hifenização perdido)
       Ex: "de de Ocor-", "Probabilidade-"
    2. Começa com hífen + lowercase (sufixo isolado)
       Ex: "-rência", "-bilidade"
    3. Repetição imediata de palavra curta (artefato OCR)
       Ex: "de de Ocor-", "do do" — palavras de 2-3 chars duplicadas seguidas
    """
    if not value:
        return False
    s = value.strip()
    # (1) hífen final solitário
    if s.endswith("-") and len(s) > 1 and not s[-2].isspace():
        return True
    # (2) hífen inicial + lowercase
    if re.match(r"^-[a-záéíóúâêôãõç]", s):
        return True
    # (3) repetição imediata de palavra curta
    if re.search(r"\b(\w{1,3})\s+\1\b", s.lower()):
        return True
    return False


def _is_column_bleed(value: str, scale: list) -> bool:
    """Texto longo demais pra ser categoria de escala (provável bleed)."""
    if not value:
        return False
    if len(value) <= _SCALE_MAX_LEN:
        return False
    # Verifica que não casa com nenhum canônico via fuzzy (se casasse alto,
    # talvez fosse só prolixo mas correto)
    _, score = fuzzy_match(value, scale, threshold=0.85)
    return score < 0.70


def _detect_column_shift(value: str, expected_scale: list, other_scales: list) -> Optional[str]:
    """Se valor casa em outra escala melhor que na esperada → column-shift.

    Retorna nome da escala onde casaria, ou None.
    """
    if not value:
        return None
    _, expected_score = fuzzy_match(value, expected_scale, threshold=0.85)
    if expected_score >= 0.85:
        return None  # casa onde deveria
    for other_name, other_scale in other_scales:
        _, other_score = fuzzy_match(value, other_scale, threshold=0.85)
        if other_score >= 0.85 and other_score > expected_score + 0.10:
            return other_name
    return None


def _audit_risk_entries(entries: List[RiskEntry]) -> Dict[str, int]:
    """Marca needs_review por padrão anômalo. Retorna contagem por categoria."""
    stats = Counter()
    field_specs = [
        ("probabilidade_original", PROBABILIDADE_SCALE, "probabilidade",
         [("impacto", IMPACTO_SCALE)]),
        ("impacto_original", IMPACTO_SCALE, "impacto",
         [("probabilidade", PROBABILIDADE_SCALE)]),
        # Simetria: detecta tratamento que na verdade é valor de outra escala
        # (column-shift). Seguro — _detect_column_shift retorna cedo quando o
        # valor já casa bem em TRATAMENTO_OPTIONS.
        ("tratamento_original", TRATAMENTO_OPTIONS, "tratamento",
         [("probabilidade", PROBABILIDADE_SCALE), ("impacto", IMPACTO_SCALE)]),
    ]
    for e in entries:
        reasons = []
        for attr, scale, label, others in field_specs:
            val = getattr(e, attr, "") or ""
            if not val:
                continue
            if _is_header_capture(val):
                reasons.append(f"{label}: header capturado ('{val[:30]}')")
                stats["header_captured"] += 1
                continue
            if _is_fragment(val):
                reasons.append(f"{label}: fragmento ('{val[:30]}')")
                stats["fragment"] += 1
                continue
            if _is_column_bleed(val, scale):
                reasons.append(f"{label}: bleed de coluna ({len(val)} chars)")
                stats["column_bleed"] += 1
                continue
            shift = _detect_column_shift(val, scale, others)
            if shift:
                reasons.append(f"{label}: column-shift (casa em '{shift}')")
                stats["column_shift"] += 1
        if reasons:
            e.needs_review = True
            existing = (e.review_reason or "").strip()
            new = "; ".join(reasons)
            e.review_reason = f"{existing}; {new}".strip("; ") if existing else new
            e.extraction_confidence = "low"
    return dict(stats)


_audit_stats = _audit_risk_entries(all_risks)
if _audit_stats:
    print("\n--- Auditoria de extração (Categoria A) ---")
    for k, v in sorted(_audit_stats.items(), key=lambda kv: -kv[1]):
        print(f"  {k:<20s} {v:>4d}")
    print(f"  TOTAL marcado needs_review por extração: {sum(_audit_stats.values())}")


organs_with_risks = set(r.orgao_sigla for r in all_risks)
organs_without_risks = set(o.sigla for o in all_organs if o.pdf_path_diretivo) - organs_with_risks
risk_errors = [e for e in all_errors if e.document_type == "diretivo" and e.stage == "extraction"]

print(f"\n{'='*60}")
print(f"RESUMO — Extração de Riscos")
print(f"{'='*60}")
print(f"  Total de riscos extraídos: {len(all_risks)}")
print(f"  Órgãos com tabela de risco: {len(organs_with_risks)}")
print(f"  Órgãos sem tabela de risco: {len(organs_without_risks)}")
if organs_without_risks:
    print(f"    → {', '.join(sorted(organs_without_risks))}")
print(f"  Erros de extração: {len(risk_errors)}")
n_with_acoes = sum(1 for r in all_risks if r.acoes_tratamento and r.acoes_tratamento.strip())
print(f"  Riscos com ações de tratamento: {n_with_acoes}")

## 6. Extração de Tabelas de Entregas (Anexos de Entregas)

Extrai entregas pactuadas via `find_tables()` com fallback textual. Captura produtos canônicos e "Outros" (projetos especiais).

In [ ]:
# ============================================================
# CÉLULA 8 — Extração de Tabelas de Entregas (PyMuPDF find_tables)
# ============================================================
# Abordagem híbrida: find_tables() para tabelas multi-página,
# fallback texto para PDFs sem estrutura tabelar.
# Captura produto "Outros" literalmente.

def _id_col(col_name: str) -> Optional[str]:
    c = _normalize_header(str(col_name))
    cs = c.strip()
    # Match exato ANTES dos substring matches — "Situação" colide com "ação" via
    # `"acao" in c` se for substring; checar status primeiro evita falso positivo.
    if cs == "status" or cs == "situacao" or cs == "situação": return "status"
    if cs == "pactuado" or cs == "pactuado?" or cs == "entregue" or cs == "entregue?": return "pactuado"
    if cs == "justificativa" or ("motivo" in c and "cancel" in c): return "justificativa"
    if "servico" in c or "acao" in c: return "servico"
    if cs in ("produto", "produto ptd", "entrega"): return "produto"
    if "eixo" in c: return "eixo"
    if "dtpactuada" in c.replace(" ","") or "data pactuada" in c or "prazo" in c: return "data_pactuada"
    if "dtentrega" in c.replace(" ","") or "data entrega" in c or "data conclusao" in c: return "data_entrega"
    return None


def _is_outros(text: str) -> bool:
    return normalize_text(text).lower().strip() in ("outros", "outro", "outros -", "outros –")


def _classify_tabela_tipo(col_map: dict, row_status: Optional[str] = None) -> str:
    """Classifica tabela como pactuada/concluida/cancelada por estrutura.

    Estratégia (defensiva, exige sinais combinados pra reduzir falso positivo):
    1. `row_status` (valor por linha de coluna 'status'/'situação') vence se vier
       um termo claro: "concluído", "cancelado", "pactuada", "em andamento".
    2. Concluida: precisa de `data_entrega` OU `pactuado` (qualquer um — colunas
       inéditas no template de pactuadas).
    3. Cancelada: precisa de `justificativa` E não pode ter `data_entrega`
       (cancelada não tem data de entrega, mas pactuada pode ter coluna
       "Justificativa" pra explicar produto "Outros").
    4. Default: "pactuada" — comportamento histórico, todos os PTDs em main.

    Especulativo: o template SGD para ciclos completos não está documentado.
    Após Colab run, se aparecerem entries com tabela_tipo != pactuada,
    inspecionar os PDFs de origem pra confirmar e ajustar.
    """
    if row_status:
        s = normalize_text(str(row_status)).lower().strip()
        if "conclu" in s or s in ("sim", "entregue", "finalizada"): return "concluida"
        if "cancel" in s or s in ("nao", "não"): return "cancelada"
        if "pactuada" in s or s == "em andamento": return "pactuada"
    if "data_entrega" in col_map or "pactuado" in col_map: return "concluida"
    if "justificativa" in col_map and "data_entrega" not in col_map and "pactuado" not in col_map:
        return "cancelada"
    return "pactuada"


_HEADER_LITERALS_DELIVERY = {
    "servico", "serviço", "servico/acao", "servico/ação", "ação", "acao",
    "produto", "produto ptd", "eixo", "data pactuada", "data entrega",
    "area responsavel", "área responsável", "id", "id entrega",
}


def _is_header_literal_delivery(value: str) -> bool:
    """True se valor é texto literal de header (capturado como dado em row)."""
    if not value:
        return False
    norm = strip_accents(normalize_text(value).lower().strip())
    return norm in _HEADER_LITERALS_DELIVERY


def _extract_deliveries_tables(pdf_path: str, sigla: str) -> List[DeliveryEntry]:
    """Extrai entregas via find_tables() com matching de produtos."""
    entries = []
    doc = fitz.open(pdf_path)
    col_map = None

    for page in doc:
        tabs = page.find_tables()
        for table in tabs.tables:
            try:
                df = table.to_pandas()
            except Exception:
                continue
            if df is None or df.empty or df.shape[1] < 3:
                continue

            # Tentar mapear colunas
            nm = {}
            for ci, col in enumerate(df.columns):
                role = _id_col(str(col))
                if role and role not in nm:
                    nm[role] = str(col)
            if "produto" in nm or ("servico" in nm and len(nm) >= 2):
                col_map = nm
            elif col_map and df.shape[1] == len(col_map):
                # Continuação multi-página: mesma estrutura, headers são dados
                # Recuperar primeira linha (que virou header de coluna)
                header_vals = [normalize_text(str(c)) for c in df.columns]
                prod_in_header = fuzzy_match_produto(header_vals[1] if len(header_vals)>1 else "")
                is_out_header = _is_outros(header_vals[1] if len(header_vals)>1 else "")
                if (prod_in_header[1] >= 0.85 or is_out_header) and header_vals[0].lower() not in ("nan","none",""):
                    # Header é dado — criar entrada e processar normalmente
                    p = prod_in_header[0] if prod_in_header[1] >= 0.85 else "Outros"
                    e = PRODUTO_TO_EIXO.get(p, "")
                    if not e and len(header_vals) > 2:
                        em = fuzzy_match_eixo(header_vals[2])
                        if em[1] >= 0.80: e = em[0]
                    entries.append(DeliveryEntry(
                        orgao_sigla=sigla, servico_acao=header_vals[0][:250],
                        produto_original=header_vals[1][:250] if len(header_vals)>1 else "",
                        produto_normalizado=p, eixo_original=header_vals[2][:100] if len(header_vals)>2 else "",
                        eixo_normalizado=e,
                        data_pactuada=parse_date(header_vals[4]) if len(header_vals)>4 and header_vals[4] else None,
                        extraction_confidence="medium", needs_review=True,
                        review_reason="recuperado de header multi-página",
                    ))
                # Usar col_map posicional existente para o restante das linhas

            if not col_map and len(df) > 0:
                for ci, val in enumerate(df.iloc[0]):
                    role = _id_col(str(val))
                    if role:
                        if col_map is None:
                            col_map = {}
                        if role not in col_map:
                            col_map[role] = str(df.columns[ci])
                if col_map:
                    df = df.iloc[1:].reset_index(drop=True)

            if not col_map:
                for _, row in df.iterrows():
                    for val in row:
                        pm = fuzzy_match_produto(str(val))
                        if pm[1] >= 0.85 or _is_outros(str(val)):
                            ci2 = list(row).index(val)
                            col_map = {"produto": str(df.columns[ci2])}
                            if ci2 > 0: col_map["servico"] = str(df.columns[ci2-1])
                            if ci2+1 < len(df.columns): col_map["eixo"] = str(df.columns[ci2+1])
                            break
                    if col_map:
                        break

            if not col_map:
                continue

            # Construir mapa posicional para tabelas de continuação
            # onde os nomes de coluna são dados, não headers
            pos_map = None
            if col_map:
                test_col = col_map.get("produto", "")
                if test_col and test_col not in df.columns:
                    # Nomes mudaram (continuação) — mapear por posição
                    # Estrutura padrão: 0=servico, 1=produto, 2=eixo, 3=area, 4=data_pactuada
                    pos_map = {"servico": 0, "produto": 1, "eixo": 2}
                    if df.shape[1] >= 5: pos_map["data_pactuada"] = 4
                    elif df.shape[1] >= 4: pos_map["data_pactuada"] = 3

            for _, row in df.iterrows():
                prod_raw = ""
                if pos_map and "produto" in pos_map:
                    prod_raw = normalize_text(str(row.iloc[pos_map["produto"]]))
                elif "produto" in col_map:
                    prod_raw = normalize_text(str(row.get(col_map["produto"], "")))
                if not prod_raw or prod_raw.lower() in ("nan", "none", "produto", "produto ptd"):
                    for val in row:
                        pm = fuzzy_match_produto(str(val))
                        if pm[1] >= 0.85 or _is_outros(str(val)):
                            prod_raw = normalize_text(str(val))
                            break
                if not prod_raw or prod_raw.lower() in ("nan", "none"):
                    continue

                # FIX ESTRUTURAL — header capturado em produto: skip row.
                # Acontece em PDFs onde find_tables pega header repetido em
                # nova página como linha de dados.
                if _is_header_literal_delivery(prod_raw):
                    continue

                pm = fuzzy_match_produto(prod_raw)
                is_out = _is_outros(prod_raw)
                if pm[1] < 0.85 and not is_out:
                    continue

                serv = ""
                if pos_map and "servico" in pos_map:
                    serv = normalize_text(str(row.iloc[pos_map["servico"]]))
                elif "servico" in col_map:
                    serv = normalize_text(str(row.get(col_map["servico"], "")))
                if serv.lower() in ("nan", "none", "servico/acao", "servico", "serviço /ação"): serv = ""

                eixo_raw = ""
                if pos_map and "eixo" in pos_map:
                    eixo_raw = normalize_text(str(row.iloc[pos_map["eixo"]]))
                elif "eixo" in col_map:
                    eixo_raw = normalize_text(str(row.get(col_map["eixo"], "")))
                if eixo_raw.lower() in ("nan", "none", "eixo"): eixo_raw = ""

                prod_norm = pm[0] if pm[1] >= 0.85 else "Outros"
                eixo_norm = PRODUTO_TO_EIXO.get(prod_norm, "")
                if not eixo_norm:
                    eixo_match = fuzzy_match_eixo(eixo_raw)
                    if eixo_match[1] >= 0.80:
                        eixo_norm = eixo_match[0]

                # Datas: pactuada e entrega são agora separadas em col_map.
                # Para retrocompat, "data" antiga vira "data_pactuada".
                def _read_field(name):
                    if pos_map and name in pos_map:
                        return normalize_text(str(row.iloc[pos_map[name]]))
                    if name in col_map:
                        return normalize_text(str(row.get(col_map[name], "")))
                    return ""

                def _clean(v):
                    return "" if v.lower() in ("nan", "none", "dtpactuada", "dtentrega") else v

                data_pact = _clean(_read_field("data_pactuada"))
                data_entr = _clean(_read_field("data_entrega"))
                pactuado_val = _clean(_read_field("pactuado"))
                justif_val = _clean(_read_field("justificativa"))
                status_val = _clean(_read_field("status"))

                tipo = _classify_tabela_tipo(col_map, status_val or None)

                entries.append(DeliveryEntry(
                    orgao_sigla=sigla,
                    tabela_tipo=tipo,
                    servico_acao=serv[:250],
                    produto_original=prod_raw[:250],
                    produto_normalizado=prod_norm,
                    eixo_original=eixo_raw,
                    eixo_normalizado=eixo_norm,
                    data_pactuada=parse_date(data_pact) if data_pact else None,
                    data_entrega=parse_date(data_entr) if data_entr else None,
                    pactuado=pactuado_val or None,
                    justificativa=justif_val[:500] if justif_val else None,
                    extraction_confidence="high" if pm[1] >= 0.95 else ("medium" if pm[1] >= 0.85 else "low"),
                    needs_review=pm[1] < 0.95 or is_out,
                    review_reason="produto Outros" if is_out else (f"fuzzy={pm[1]:.2f}" if pm[1] < 0.95 else None),
                ))

    doc.close()
    return entries


def _extract_deliveries_text(pdf_path: str, sigla: str) -> List[DeliveryEntry]:
    """Fallback: extrai entregas por matching de produto no texto linha-a-linha."""
    entries = []
    doc = fitz.open(pdf_path)

    for page in doc:
        lines = [l.strip() for l in page.get_text().split('\n') if l.strip()]
        for i, line in enumerate(lines):
            pm = fuzzy_match_produto(line)
            is_out = _is_outros(line)
            if pm[1] < 0.85 and not is_out:
                continue

            serv = ""
            if i > 0:
                prev = lines[i-1]
                if (fuzzy_match_produto(prev)[1] < 0.80 and
                    fuzzy_match_eixo(prev)[1] < 0.80 and len(prev) > 5 and
                    not re.match(r"^(jan|fev|mar|abr|mai|jun|jul|ago|set|out|nov|dez)", prev.lower())):
                    serv = prev[:250]

            prod_norm = pm[0] if pm[1] >= 0.85 else "Outros"
            eixo_norm = PRODUTO_TO_EIXO.get(prod_norm, "")
            if not eixo_norm and i+1 < len(lines):
                em = fuzzy_match_eixo(lines[i+1])
                if em[1] >= 0.80:
                    eixo_norm = em[0]

            data = ""
            for j in range(i+1, min(i+5, len(lines))):
                if re.match(r"^(jan|fev|mar|abr|mai|jun|jul|ago|set|out|nov|dez)[\-/]\d{2,4}$", lines[j].lower()):
                    data = lines[j]; break
                if re.match(r"^\d{2}/\d{2,4}$", lines[j]):
                    data = lines[j]; break

            entries.append(DeliveryEntry(
                orgao_sigla=sigla,
                servico_acao=serv,
                produto_original=line[:250],
                produto_normalizado=prod_norm,
                eixo_original="",
                eixo_normalizado=eixo_norm,
                data_pactuada=parse_date(data) if data else None,
                extraction_confidence="high" if pm[1] >= 0.95 else ("medium" if pm[1] >= 0.85 else "low"),
                needs_review=pm[1] < 0.95 or is_out,
                review_reason="produto Outros" if is_out else None,
            ))

    doc.close()
    return entries


# --------------- Extração em lote (híbrida) -------------------

def extract_all_deliveries() -> None:
    global all_deliveries, all_errors

    # Fingerprint do estado upstream (siglas com PDF entregas após dedup MD5).
    # Invalida o cache automaticamente quando 05c_dedup zera paths.
    fp = state_fingerprint(sorted((o.sigla, bool(o.pdf_path_entregas)) for o in all_organs))

    cached = load_checkpoint("deliveries_raw", expected_fingerprint=fp)
    if cached is not None and len(cached[0]) > 0:
        cached_del, cached_errors, processed_siglas = cached
        all_deliveries.extend(cached_del)
        all_errors.extend(cached_errors)
        print(f"  Retomando: {len(cached_del)} entregas de {len(processed_siglas)} órgãos")
    else:
        cached_del, cached_errors, processed_siglas = [], [], set()

    organs_with_pdf = [o for o in all_organs if o.pdf_path_entregas]
    pending = [o for o in organs_with_pdf if o.sigla not in processed_siglas]

    if not pending:
        print("  Todos os órgãos já processados (checkpoint).")
        return

    print(f"  Processando: {len(pending)} órgãos pendentes")

    pdf_results_cache: Dict[str, Tuple[List[DeliveryEntry], str]] = {}
    batch_del, batch_errors = [], []
    count = 0

    for organ in tqdm(pending, desc="Extraindo entregas"):
        sigla = organ.sigla
        pdf_path = organ.pdf_path_entregas

        if not os.path.isfile(pdf_path):
            batch_errors.append(ProcessingError(orgao_sigla=sigla, document_type="entregas",
                stage="extraction", error_type="file_not_found",
                error_message=f"PDF não encontrado: {pdf_path}"))
            processed_siglas.add(sigla)
            count += 1
            continue

        real_path = os.path.realpath(pdf_path)
        if real_path in pdf_results_cache:
            owner = pdf_results_cache[real_path][1]
            processed_siglas.add(sigla)
            logger.info(f"[{sigla}] PDF compartilhado com {owner} — sem duplicação")
        else:
            # Híbrido: find_tables primeiro, fallback texto
            ft_entries = _extract_deliveries_tables(pdf_path, sigla)
            tx_entries = _extract_deliveries_text(pdf_path, sigla)
            best = ft_entries if len(ft_entries) >= len(tx_entries) else tx_entries

            pdf_results_cache[real_path] = (best, sigla)
            batch_del.extend(best)
            all_deliveries.extend(best)
            processed_siglas.add(sigla)

            if best:
                logger.info(f"[{sigla}] {len(best)} entregas extraídas")

        count += 1
        if count % 10 == 0:
            save_checkpoint((cached_del + batch_del, cached_errors + batch_errors, processed_siglas), "deliveries_raw", fingerprint=fp)

    save_checkpoint((cached_del + batch_del, cached_errors + batch_errors, processed_siglas), "deliveries_raw", fingerprint=fp)
    print(f"  Extração de entregas concluída.")


# --------------- Execução -------------------------------------
extract_all_deliveries()

organs_with_del = set(d.orgao_sigla for d in all_deliveries)
del_errors = [e for e in all_errors if e.document_type == "entregas" and e.stage == "extraction"]

print(f"\n{'='*60}")
print(f"RESUMO — Extração de Entregas")
print(f"{'='*60}")
print(f"  Total de entregas extraídas: {len(all_deliveries)}")
print(f"  Órgãos com entregas: {len(organs_with_del)}")
print(f"  Erros de extração: {len(del_errors)}")

tipo_counts = {}
for d in all_deliveries:
    t = d.tabela_tipo or "pactuada"
    tipo_counts[t] = tipo_counts.get(t, 0) + 1
print(f"  Por tipo: {tipo_counts}")

n_outros = sum(1 for d in all_deliveries if d.produto_normalizado == "Outros")
print(f"  Produto 'Outros': {n_outros}")

## 7. Padronização de Vocabulário

Normaliza produtos, eixos, probabilidades, impactos e tratamentos usando matching em camadas.

In [ ]:
# ============================================================
# CÉLULA 9 — Padronização de Vocabulário
# ============================================================
from typing import List, Tuple, Dict
from collections import Counter


def standardize_deliveries(entries: List[DeliveryEntry]) -> Tuple[List[DeliveryEntry], Dict]:
    """
    Normaliza produto e eixo de cada DeliveryEntry usando matching em camadas.

    Retorna a lista atualizada e um relatório de vocabulário.
    """
    produto_mappings: Dict[str, Dict] = {}   # original -> {normalized, score, count}
    eixo_mappings: Dict[str, Dict] = {}
    unmatched_produtos: List[str] = []
    stats = Counter()  # exact, fuzzy, unmatched

    for entry in entries:
        # --- Produto ---
        prod_orig = normalize_text(entry.produto_original)

        if prod_orig in produto_mappings:
            cached = produto_mappings[prod_orig]
            entry.produto_normalizado = cached["normalized"]
            cached["count"] += 1
            score = cached["score"]
            entry.produto_score = round(score, 3)
            entry.produto_method = cached["method"]
        else:
            matched, score = fuzzy_match_produto(entry.produto_original)
            if score >= 0.85:
                entry.produto_normalizado = matched
            elif score >= 0.70:
                entry.produto_normalizado = matched
                entry.needs_review = True
                entry.review_reason = f"fuzzy match baixo ({score:.2f})"
                entry.extraction_confidence = "medium"
            else:
                entry.produto_normalizado = "UNMATCHED"
                entry.needs_review = True
                entry.review_reason = f"produto não reconhecido (melhor score: {score:.2f})"
                entry.extraction_confidence = "low"

            entry.produto_score = round(score, 3)
            entry.produto_method = classify_match(entry.produto_original, score, PRODUTO_ALIASES)

            produto_mappings[prod_orig] = {
                "normalized": entry.produto_normalizado,
                "score": score,
                "method": entry.produto_method,
                "count": 1,
            }

        # Track stats
        if score == 1.0:
            stats["exact"] += 1
        elif score >= 0.70:
            stats["fuzzy"] += 1
        else:
            stats["unmatched"] += 1

        # --- Eixo ---
        eixo_orig = normalize_text(entry.eixo_original)

        if eixo_orig in eixo_mappings:
            cached_eixo = eixo_mappings[eixo_orig]
            entry.eixo_normalizado = cached_eixo["normalized"]
            cached_eixo["count"] += 1
            entry.eixo_score = round(cached_eixo["score"], 3)
            entry.eixo_method = cached_eixo["method"]
        else:
            matched_eixo, eixo_score = fuzzy_match_eixo(entry.eixo_original)
            if eixo_score >= 0.70:
                entry.eixo_normalizado = matched_eixo
            else:
                entry.eixo_normalizado = entry.eixo_original  # keep original
            entry.eixo_score = round(eixo_score, 3)
            entry.eixo_method = classify_match(entry.eixo_original, eixo_score, EIXO_ALIASES)
            eixo_mappings[eixo_orig] = {
                "normalized": entry.eixo_normalizado,
                "score": eixo_score,
                "method": entry.eixo_method,
                "count": 1,
            }

        # --- Cross-validation: produto ↔ eixo ---
        if entry.produto_normalizado != "UNMATCHED" and entry.produto_normalizado in PRODUTO_TO_EIXO:
            canonical_eixo = PRODUTO_TO_EIXO[entry.produto_normalizado]
            if entry.eixo_normalizado != canonical_eixo:
                # Prefer the canonical mapping from the product
                old_eixo = entry.eixo_normalizado
                entry.eixo_normalizado = canonical_eixo
                if not entry.needs_review:
                    entry.needs_review = True
                    entry.review_reason = (
                        f"eixo corrigido por cross-validation: "
                        f"'{old_eixo}' → '{canonical_eixo}'"
                    )
                elif entry.review_reason and "eixo corrigido" not in entry.review_reason:
                    entry.review_reason += (
                        f"; eixo corrigido: '{old_eixo}' → '{canonical_eixo}'"
                    )

    # Build unmatched list
    unmatched_produtos = [
        orig for orig, info in produto_mappings.items()
        if info["normalized"] == "UNMATCHED"
    ]

    vocab_report = {
        "produto_mappings": [
            {
                "original": orig,
                "normalized": info["normalized"],
                "score": info["score"],
                "count": info["count"],
            }
            for orig, info in sorted(produto_mappings.items())
        ],
        "eixo_mappings": [
            {
                "original": orig,
                "normalized": info["normalized"],
                "score": info["score"],
                "count": info["count"],
            }
            for orig, info in sorted(eixo_mappings.items())
        ],
        "unmatched_produtos": unmatched_produtos,
        "match_stats": dict(stats),
    }

    return entries, vocab_report


def filter_fragment_deliveries(
        entries: List[DeliveryEntry]) -> Tuple[List[DeliveryEntry], List[DeliveryEntry]]:
    """Descarta fragmentos de extração: 'Outros' sem descrição substantiva.

    Células quebradas por página/linha geram registros onde o produto não
    casa com o catálogo (vira 'Outros') e servico_acao fica vazio ou com um
    resto da célula anterior (ex.: 'Meu RPPS', 'rência'). Projetos Especiais
    legítimos têm sempre descrição substantiva — o corte em <10 chars
    preserva o texto livre validado pela curadoria (2026-06).

    Retorna (mantidas, descartadas). Aplicar APÓS standardize_deliveries,
    que é quem define produto_normalizado.
    """
    kept: List[DeliveryEntry] = []
    dropped: List[DeliveryEntry] = []
    for e in entries:
        sa = (e.servico_acao or "").strip()
        if e.produto_normalizado == "Outros" and len(sa) < 10:
            dropped.append(e)
        else:
            kept.append(e)
    return kept, dropped


def standardize_risks(entries: List[RiskEntry]) -> Tuple[List[RiskEntry], Dict]:
    """
    Normaliza probabilidade, impacto e tratamento de cada RiskEntry.

    Retorna a lista atualizada e um relatório de normalização.
    """
    prob_mappings: Dict[str, Dict] = {}
    imp_mappings: Dict[str, Dict] = {}
    trat_mappings: Dict[str, Dict] = {}
    stats = {
        "probabilidade": Counter(),  # exact, fuzzy, unmatched
        "impacto": Counter(),
        "tratamento": Counter(),
    }

    for entry in entries:
        review_reasons = []

        # --- Probabilidade ---
        prob_orig = normalize_text(entry.probabilidade_original)
        if prob_orig in prob_mappings:
            cached = prob_mappings[prob_orig]
            entry.probabilidade_normalizada = cached["normalized"]
            cached["count"] += 1
            p_score = cached["score"]
            entry.probabilidade_score = round(p_score, 3)
            entry.probabilidade_method = cached["method"]
        else:
            matched, p_score = fuzzy_match_scale(entry.probabilidade_original, PROBABILIDADE_SCALE)
            if p_score >= 0.85:
                entry.probabilidade_normalizada = matched
            elif p_score >= 0.70:
                entry.probabilidade_normalizada = matched
                review_reasons.append(f"probabilidade fuzzy ({p_score:.2f})")
            else:
                entry.probabilidade_normalizada = entry.probabilidade_original
                review_reasons.append(f"probabilidade não reconhecida (score: {p_score:.2f})")
            entry.probabilidade_score = round(p_score, 3)
            entry.probabilidade_method = classify_match(
                entry.probabilidade_original, p_score, PROBABILIDADE_ALIASES)
            prob_mappings[prob_orig] = {
                "normalized": entry.probabilidade_normalizada,
                "score": p_score,
                "method": entry.probabilidade_method,
                "count": 1,
            }

        if p_score == 1.0:
            stats["probabilidade"]["exact"] += 1
        elif p_score >= 0.70:
            stats["probabilidade"]["fuzzy"] += 1
        else:
            stats["probabilidade"]["unmatched"] += 1

        # --- Impacto ---
        imp_orig = normalize_text(entry.impacto_original)
        if imp_orig in imp_mappings:
            cached = imp_mappings[imp_orig]
            entry.impacto_normalizado = cached["normalized"]
            cached["count"] += 1
            i_score = cached["score"]
            entry.impacto_score = round(i_score, 3)
            entry.impacto_method = cached["method"]
        else:
            matched, i_score = fuzzy_match_scale(entry.impacto_original, IMPACTO_SCALE)
            if i_score >= 0.85:
                entry.impacto_normalizado = matched
            elif i_score >= 0.70:
                entry.impacto_normalizado = matched
                review_reasons.append(f"impacto fuzzy ({i_score:.2f})")
            else:
                entry.impacto_normalizado = entry.impacto_original
                review_reasons.append(f"impacto não reconhecido (score: {i_score:.2f})")
            entry.impacto_score = round(i_score, 3)
            entry.impacto_method = classify_match(
                entry.impacto_original, i_score, IMPACTO_ALIASES)
            imp_mappings[imp_orig] = {
                "normalized": entry.impacto_normalizado,
                "score": i_score,
                "method": entry.impacto_method,
                "count": 1,
            }

        if i_score == 1.0:
            stats["impacto"]["exact"] += 1
        elif i_score >= 0.70:
            stats["impacto"]["fuzzy"] += 1
        else:
            stats["impacto"]["unmatched"] += 1

        # --- Tratamento (pode ser múltiplo: separado por ; , /) ---
        trat_orig = normalize_text(entry.tratamento_original)
        if trat_orig in trat_mappings:
            cached = trat_mappings[trat_orig]
            entry.tratamento_normalizado = cached["normalized"]
            cached["count"] += 1
            t_score = cached["score"]
            entry.tratamento_score = round(t_score, 3)
            entry.tratamento_method = cached["method"]
        else:
            # Split by multiple separators
            parts = re.split(r"\s*[;,/]\s*", trat_orig) if trat_orig else []
            normalized_parts = []
            worst_score = 1.0

            for part in parts:
                part = part.strip()
                if not part:
                    continue
                matched, t_sc = fuzzy_match_scale(part, TRATAMENTO_OPTIONS)
                if t_sc >= 0.70:
                    normalized_parts.append(matched)
                else:
                    normalized_parts.append(part)
                    review_reasons.append(f"tratamento '{part}' não reconhecido (score: {t_sc:.2f})")
                worst_score = min(worst_score, t_sc)

            entry.tratamento_normalizado = "; ".join(normalized_parts) if normalized_parts else trat_orig
            t_score = worst_score if parts else 0.0
            entry.tratamento_score = round(t_score, 3)
            # Para tratamento multi-valor, classifica pelo PIOR caso (worst_score)
            # contra o alias map de tratamento. O original aqui é a string completa.
            entry.tratamento_method = classify_match(
                entry.tratamento_original, t_score, TRATAMENTO_ALIASES)
            trat_mappings[trat_orig] = {
                "normalized": entry.tratamento_normalizado,
                "score": t_score,
                "method": entry.tratamento_method,
                "count": 1,
            }

        if t_score == 1.0:
            stats["tratamento"]["exact"] += 1
        elif t_score >= 0.70:
            stats["tratamento"]["fuzzy"] += 1
        else:
            stats["tratamento"]["unmatched"] += 1

        # Tratamento composto/múltiplo (ex.: "mitigar; transferir"): o template SGD
        # prevê UMA opção por risco. Múltiplas partes indicam bleed de coluna ou
        # escolha ambígua e merecem revisão humana — mesmo quando cada parte casa
        # bem (worst_score alto). Não altera tratamento_normalizado.
        if "; " in (entry.tratamento_normalizado or ""):
            entry.needs_review = True
            review_reasons.append(
                f"tratamento múltiplo/composto ('{entry.tratamento_normalizado[:40]}')")

        # --- Set confidence and review flags ---
        scores = [p_score, i_score, t_score]
        min_score = min(scores) if scores else 0.0

        if min_score >= 0.85:
            if not entry.needs_review:
                entry.extraction_confidence = "high"
        elif min_score >= 0.70:
            entry.extraction_confidence = "medium"
            entry.needs_review = True
        else:
            entry.extraction_confidence = "low"
            entry.needs_review = True

        # Qualquer motivo de revisão registrado implica needs_review — fecha a
        # lacuna em que um review_reason era anexado sem sinalizar a linha.
        if review_reasons:
            entry.needs_review = True
            existing = entry.review_reason or ""
            new_reasons = "; ".join(review_reasons)
            entry.review_reason = f"{existing}; {new_reasons}".strip("; ") if existing else new_reasons

    risk_report = {
        "probabilidade_mappings": [
            {"original": orig, "normalized": info["normalized"], "score": info["score"], "count": info["count"]}
            for orig, info in sorted(prob_mappings.items())
        ],
        "impacto_mappings": [
            {"original": orig, "normalized": info["normalized"], "score": info["score"], "count": info["count"]}
            for orig, info in sorted(imp_mappings.items())
        ],
        "tratamento_mappings": [
            {"original": orig, "normalized": info["normalized"], "score": info["score"], "count": info["count"]}
            for orig, info in sorted(trat_mappings.items())
        ],
        "stats": {k: dict(v) for k, v in stats.items()},
    }

    return entries, risk_report


# ---- Execução ----
print("=" * 60)
print("PADRONIZAÇÃO DE VOCABULÁRIO")
print("=" * 60)

# Standardize deliveries
if all_deliveries:
    print(f"\nPadronizando {len(all_deliveries)} entregas...")
    all_deliveries, vocab_report = standardize_deliveries(all_deliveries)

    d_stats = vocab_report["match_stats"]
    d_total = sum(d_stats.values()) or 1
    print(f"\n  Produtos — Match exato: {d_stats.get('exact', 0)} "
          f"({d_stats.get('exact', 0)/d_total*100:.1f}%) | "
          f"Fuzzy: {d_stats.get('fuzzy', 0)} "
          f"({d_stats.get('fuzzy', 0)/d_total*100:.1f}%) | "
          f"Não reconhecidos: {d_stats.get('unmatched', 0)} "
          f"({d_stats.get('unmatched', 0)/d_total*100:.1f}%)")

    print(f"  Termos únicos de produto: {len(vocab_report['produto_mappings'])}")
    print(f"  Termos únicos de eixo:    {len(vocab_report['eixo_mappings'])}")

    if vocab_report["unmatched_produtos"]:
        print(f"\n  Produtos NÃO RECONHECIDOS ({len(vocab_report['unmatched_produtos'])}):")
        for p in vocab_report["unmatched_produtos"][:20]:
            print(f"    - '{p}'")
        if len(vocab_report["unmatched_produtos"]) > 20:
            print(f"    ... e mais {len(vocab_report['unmatched_produtos']) - 20}")

    # Remoção de fragmentos de célula (depende de produto_normalizado)
    all_deliveries, _fragmentos = filter_fragment_deliveries(all_deliveries)
    if _fragmentos:
        _frag_orgaos = Counter(e.orgao_sigla for e in _fragmentos)
        print(f"\n  Fragmentos descartados ('Outros' + servico_acao <10 chars): "
              f"{len(_fragmentos)}")
        print("    " + ", ".join(f"{o} ({n})" for o, n in _frag_orgaos.most_common()))
else:
    print("\nNenhuma entrega para padronizar.")
    vocab_report = {
        "produto_mappings": [],
        "eixo_mappings": [],
        "unmatched_produtos": [],
        "match_stats": {"exact": 0, "fuzzy": 0, "unmatched": 0},
    }

# Standardize risks
if all_risks:
    print(f"\nPadronizando {len(all_risks)} riscos...")
    all_risks, risk_report = standardize_risks(all_risks)

    for field_name in ["probabilidade", "impacto", "tratamento"]:
        fstats = risk_report["stats"].get(field_name, {})
        ftotal = sum(fstats.values()) or 1
        print(f"\n  {field_name.capitalize():<16s} — "
              f"Exato: {fstats.get('exact', 0):4d} ({fstats.get('exact', 0)/ftotal*100:.1f}%) | "
              f"Fuzzy: {fstats.get('fuzzy', 0):4d} ({fstats.get('fuzzy', 0)/ftotal*100:.1f}%) | "
              f"Falhou: {fstats.get('unmatched', 0):4d} ({fstats.get('unmatched', 0)/ftotal*100:.1f}%)")

    print(f"\n  Termos únicos — probabilidade: {len(risk_report['probabilidade_mappings'])}, "
          f"impacto: {len(risk_report['impacto_mappings'])}, "
          f"tratamento: {len(risk_report['tratamento_mappings'])}")
else:
    print("\nNenhum risco para padronizar.")
    risk_report = {
        "probabilidade_mappings": [],
        "impacto_mappings": [],
        "tratamento_mappings": [],
        "stats": {},
    }

# Review summary
n_review_del = sum(1 for d in all_deliveries if d.needs_review)
n_review_risk = sum(1 for r in all_risks if r.needs_review)
print(f"\n{'='*60}")
print(f"RESUMO DA PADRONIZAÇÃO")
print(f"  Entregas para revisão: {n_review_del}/{len(all_deliveries)}")
print(f"  Riscos para revisão:   {n_review_risk}/{len(all_risks)}")
print(f"{'='*60}")

# Save checkpoints — fingerprints derivados das contagens permitem que cells
# downstream (caso venham a ler estes pickles) detectem cache pré-dedup.
_fp_del_std = state_fingerprint((len(all_deliveries),
                                 sorted({d.orgao_sigla for d in all_deliveries})))
_fp_risk_std = state_fingerprint((len(all_risks),
                                  sorted({r.orgao_sigla for r in all_risks})))
save_checkpoint(all_deliveries, "deliveries_standardized", fingerprint=_fp_del_std)
save_checkpoint(all_risks, "risks_standardized", fingerprint=_fp_risk_std)
save_checkpoint(vocab_report, "vocab_report")
save_checkpoint(risk_report, "risk_report")
print("\nCheckpoints de padronização salvos.")

## 8. Exportação de Dados

Exporta dados processados em CSV e JSON, mais relatório de erros e ajustes.

In [ ]:
# ============================================================
# CÉLULA 10 — Exportação de Dados
# ============================================================
from datetime import datetime
from dataclasses import asdict


def _file_size_str(path: str) -> str:
    """Retorna tamanho do arquivo em formato legível."""
    size = os.path.getsize(path)
    if size < 1024:
        return f"{size} B"
    elif size < 1024 * 1024:
        return f"{size / 1024:.1f} KB"
    else:
        return f"{size / (1024 * 1024):.1f} MB"


def _build_nested_json(entries: list, key_field: str, metadata_extra: dict = None) -> dict:
    """Agrupa entradas por key_field em um JSON com metadata.

    `entries` deve vir ordenado pelo chamador para que `grouped` resulte
    determinístico (Python 3.7+ preserva ordem de inserção em dict).
    """
    grouped = {}
    for entry in entries:
        d = asdict(entry)
        key = d.get(key_field, "UNKNOWN")
        if key not in grouped:
            grouped[key] = []
        grouped[key].append(d)

    metadata = {
        "exported_at": datetime.now().isoformat(),
        "total": len(entries),
        "groups": len(grouped),
    }
    if metadata_extra:
        metadata.update(metadata_extra)

    return {"metadata": metadata, "data": grouped}


def _sorted_stable(entries: list, *keys):
    """Ordena por tuplas de chaves, tolerando None (vira string vazia).

    Usa sort estável (Timsort) para preservar a ordem upstream entre
    registros equivalentes.
    """
    def _key(e):
        d = asdict(e) if hasattr(e, "__dataclass_fields__") else e
        return tuple((d.get(k) or "") for k in keys)
    return sorted(entries, key=_key)


export_log = []  # (filename, rows, size_str)

# Ordenação estável aplicada in-place para garantir reprodutibilidade
# bit-a-bit dos CSVs/JSONs entre runs (sem depender da ordem upstream).
all_deliveries = _sorted_stable(
    all_deliveries, "orgao_sigla", "eixo_normalizado",
    "produto_normalizado", "servico_acao",
)
all_risks = _sorted_stable(all_risks, "orgao_sigla", "risco_texto")
all_organs = _sorted_stable(all_organs, "sigla")
all_errors = _sorted_stable(all_errors, "orgao_sigla", "document_type", "stage")

# ---- 1. Entregas: CSV e JSON ----
if all_deliveries:
    df_del = pd.DataFrame([asdict(e) for e in all_deliveries])

    csv_path = os.path.join(DIRS["output"], "deliveries.csv")
    df_del.to_csv(csv_path, index=False, encoding="utf-8-sig")
    export_log.append(("deliveries.csv", len(df_del), _file_size_str(csv_path)))

    json_path = os.path.join(DIRS["output"], "deliveries.json")
    nested = _build_nested_json(all_deliveries, "orgao_sigla")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(nested, f, ensure_ascii=False, indent=2)
    export_log.append(("deliveries.json", len(all_deliveries), _file_size_str(json_path)))
else:
    print("Nenhuma entrega para exportar.")

# ---- 2. Riscos: CSV e JSON ----
if all_risks:
    df_risk = pd.DataFrame([asdict(e) for e in all_risks])

    csv_path = os.path.join(DIRS["output"], "risks.csv")
    df_risk.to_csv(csv_path, index=False, encoding="utf-8-sig")
    export_log.append(("risks.csv", len(df_risk), _file_size_str(csv_path)))

    json_path = os.path.join(DIRS["output"], "risks.json")
    nested = _build_nested_json(all_risks, "orgao_sigla")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(nested, f, ensure_ascii=False, indent=2)
    export_log.append(("risks.json", len(all_risks), _file_size_str(json_path)))
else:
    print("Nenhum risco para exportar.")

# ---- 3. Órgãos: CSV ----
if all_organs:
    df_org = pd.DataFrame([asdict(o) for o in all_organs])

    csv_path = os.path.join(DIRS["output"], "organs.csv")
    df_org.to_csv(csv_path, index=False, encoding="utf-8-sig")
    export_log.append(("organs.csv", len(df_org), _file_size_str(csv_path)))
else:
    print("Nenhum órgão para exportar.")

# ---- 4. Relatório de erros: CSV ----
if all_errors:
    df_err = pd.DataFrame([asdict(e) for e in all_errors])

    csv_path = os.path.join(DIRS["output"], "error_report.csv")
    df_err.to_csv(csv_path, index=False, encoding="utf-8-sig")
    export_log.append(("error_report.csv", len(df_err), _file_size_str(csv_path)))
else:
    print("Nenhum erro registrado para exportar.")

# ---- 5. Mapeamento de vocabulário: CSV ----
vocab_rows = []

# Produto mappings
for m in vocab_report.get("produto_mappings", []):
    vocab_rows.append({
        "type": "produto",
        "original": m["original"],
        "normalized": m["normalized"],
        "score": m["score"],
        "count": m["count"],
    })

# Eixo mappings
for m in vocab_report.get("eixo_mappings", []):
    vocab_rows.append({
        "type": "eixo",
        "original": m["original"],
        "normalized": m["normalized"],
        "score": m["score"],
        "count": m["count"],
    })

# Risk field mappings
for field_key in ["probabilidade_mappings", "impacto_mappings", "tratamento_mappings"]:
    field_type = field_key.replace("_mappings", "")
    for m in risk_report.get(field_key, []):
        vocab_rows.append({
            "type": field_type,
            "original": m["original"],
            "normalized": m["normalized"],
            "score": m["score"],
            "count": m["count"],
        })

# Sempre grava o arquivo (mesmo vazio com header) para alinhar com o
# README e tornar a ausência de mapeamentos uma sentinela explícita.
csv_path = os.path.join(DIRS["output"], "vocabulary_mapping.csv")
vocab_cols = ["type", "original", "normalized", "score", "count"]
df_vocab = pd.DataFrame(vocab_rows, columns=vocab_cols)
if not df_vocab.empty:
    df_vocab = df_vocab.sort_values(
        ["type", "original"], kind="mergesort"
    ).reset_index(drop=True)
df_vocab.to_csv(csv_path, index=False, encoding="utf-8-sig")
export_log.append(("vocabulary_mapping.csv", len(df_vocab), _file_size_str(csv_path)))
if df_vocab.empty:
    print("vocabulary_mapping.csv gravado vazio (nenhum mapeamento aplicado).")

# ---- Resumo de exportação ----
print("\n" + "=" * 60)
print("RESUMO DA EXPORTAÇÃO")
print("=" * 60)
print(f"Diretório de saída: {DIRS['output']}\n")
print(f"{'Arquivo':<30s} {'Registros':>10s} {'Tamanho':>10s}")
print("-" * 52)
for fname, rows, size in export_log:
    print(f"{fname:<30s} {rows:>10,d} {size:>10s}")
print("-" * 52)
total_files = len(export_log)
total_rows = sum(r for _, r, _ in export_log)
print(f"{'TOTAL':<30s} {total_rows:>10,d}   ({total_files} arquivos)")
print("=" * 60)

## Célula 13 — Relatório de Validação do Ciclo

Exporta um único JSON (`output/validation_report.json`) consolidando todas as métricas que precisam ser auditadas após cada run do pipeline. Útil para:

- **Validação pós-mudança**: comparar antes/depois ao introduzir aliases (Camada 1.5), corrigir extração tabular (Categoria A) ou refatorar utilitários.
- **Auditoria entre máquinas**: confirmar que um run no Colab produziu o mesmo resultado que rodaria localmente (checksums dos CSVs).
- **Histórico de regressões**: arquivar o JSON após cada run significativo cria uma série temporal de saúde do pipeline.

O JSON contém: contagens, taxas de canonização, top-20 residuais por campo, status dos thresholds de qualidade e MD5 dos outputs principais.

In [ ]:
# ============================================================
# CÉLULA 13 — Exporta validation_report.json
# ============================================================
import json, hashlib
from collections import Counter
from datetime import datetime, timezone

def _md5_file(path: str) -> str:
    if not os.path.exists(path):
        return ""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def _topn(values, n=20):
    return [{"value": v, "count": c} for v, c in Counter(values).most_common(n)]

# --- Contagens básicas ---
n_entregas = len(all_deliveries) if all_deliveries else 0
n_riscos = len(all_risks) if all_risks else 0
n_orgaos = len(all_organs) if all_organs else 0

# --- Taxas de canonização e residuais ---
prob_ok = imp_ok = trat_ok = 0
nc_prob_vals, nc_imp_vals, nc_trat_vals = [], [], []
if all_risks:
    for r in all_risks:
        if r.probabilidade_normalizada in PROBABILIDADE_SCALE:
            prob_ok += 1
        elif r.probabilidade_original:
            nc_prob_vals.append(r.probabilidade_original)
        if r.impacto_normalizado in IMPACTO_SCALE:
            imp_ok += 1
        elif r.impacto_original:
            nc_imp_vals.append(r.impacto_original)
        trat_canonical = (r.tratamento_normalizado and
                          all(t.strip() in TRATAMENTO_OPTIONS
                              for t in r.tratamento_normalizado.split(";") if t.strip()))
        if trat_canonical:
            trat_ok += 1
        elif r.tratamento_original:
            nc_trat_vals.append(r.tratamento_original)

prob_rate = prob_ok / n_riscos if n_riscos else 0.0
imp_rate = imp_ok / n_riscos if n_riscos else 0.0
trat_rate = trat_ok / n_riscos if n_riscos else 0.0

# --- Threshold checks ---
thresholds = {
    "max_entregas": {
        "limit": QUALITY_THRESHOLDS["max_entregas"],
        "actual": n_entregas,
        "passed": n_entregas <= QUALITY_THRESHOLDS["max_entregas"],
    },
    "max_riscos": {
        "limit": QUALITY_THRESHOLDS["max_riscos"],
        "actual": n_riscos,
        "passed": n_riscos <= QUALITY_THRESHOLDS["max_riscos"],
    },
    "min_prob_canonica_ratio": {
        "limit": QUALITY_THRESHOLDS["min_prob_canonica_ratio"],
        "actual": round(prob_rate, 4),
        "passed": prob_rate >= QUALITY_THRESHOLDS["min_prob_canonica_ratio"],
    },
    "min_imp_canonica_ratio": {
        "limit": QUALITY_THRESHOLDS["min_imp_canonica_ratio"],
        "actual": round(imp_rate, 4),
        "passed": imp_rate >= QUALITY_THRESHOLDS["min_imp_canonica_ratio"],
    },
    "min_trat_canonica_ratio": {
        "limit": QUALITY_THRESHOLDS["min_trat_canonica_ratio"],
        "actual": round(trat_rate, 4),
        "passed": trat_rate >= QUALITY_THRESHOLDS["min_trat_canonica_ratio"],
    },
}

# --- Needs-review ---
n_review_del = sum(1 for d in all_deliveries if d.needs_review) if all_deliveries else 0
n_review_risk = sum(1 for r in all_risks if r.needs_review) if all_risks else 0

# --- Distribuições normalizadas (para detectar shifts) ---
dist_prob = Counter(r.probabilidade_normalizada for r in all_risks) if all_risks else {}
dist_imp = Counter(r.impacto_normalizado for r in all_risks) if all_risks else {}
dist_trat = Counter(r.tratamento_normalizado for r in all_risks) if all_risks else {}

# --- Checksums dos outputs ---
out = DIRS["output"]
checksums = {
    name: _md5_file(os.path.join(out, name))
    for name in ["risks.csv", "deliveries.csv", "organs.csv",
                 "risks.json", "deliveries.json", "error_report.csv"]
}

# --- Pipeline fingerprints (um por checkpoint, gravados por save_checkpoint) ---
fingerprints = {}
ckpt_dir = DIRS.get("checkpoints", "")
if ckpt_dir and os.path.isdir(ckpt_dir):
    for fname in sorted(os.listdir(ckpt_dir)):
        if fname.endswith(".fingerprint"):
            try:
                with open(os.path.join(ckpt_dir, fname), encoding="utf-8") as f:
                    fingerprints[fname[:-len(".fingerprint")]] = f.read().strip()
            except Exception:
                pass

report = {
    "schema_version": 1,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "counts": {
        "orgaos": n_orgaos,
        "entregas": n_entregas,
        "riscos": n_riscos,
        "errors": len(all_errors) if all_errors else 0,
        "needs_review_entregas": n_review_del,
        "needs_review_riscos": n_review_risk,
    },
    "canonization": {
        "probabilidade": {"ok": prob_ok, "total": n_riscos, "rate": round(prob_rate, 4)},
        "impacto":       {"ok": imp_ok,  "total": n_riscos, "rate": round(imp_rate, 4)},
        "tratamento":    {"ok": trat_ok, "total": n_riscos, "rate": round(trat_rate, 4)},
    },
    "non_canonical_top20": {
        "probabilidade": _topn(nc_prob_vals),
        "impacto":       _topn(nc_imp_vals),
        "tratamento":    _topn(nc_trat_vals),
    },
    "distributions_normalized": {
        "probabilidade": dict(dist_prob),
        "impacto":       dict(dist_imp),
        "tratamento":    dict(dist_trat),
    },
    "thresholds": thresholds,
    "all_thresholds_passed": all(t["passed"] for t in thresholds.values()),
    "output_checksums_md5": checksums,
    "checkpoint_fingerprints": fingerprints,
}

report_path = os.path.join(out, "validation_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("=" * 60)
print("RELATÓRIO DE VALIDAÇÃO")
print("=" * 60)
print(f"  Arquivo: {report_path}")
print(f"  Entregas: {n_entregas}  |  Riscos: {n_riscos}  |  Órgãos: {n_orgaos}")
print(f"  Canonização: prob={prob_rate:.1%} imp={imp_rate:.1%} trat={trat_rate:.1%}")
print(f"  Residuais: prob={len(set(nc_prob_vals))} imp={len(set(nc_imp_vals))} trat={len(set(nc_trat_vals))} valores únicos")
print(f"  Thresholds: {'✓ todos passaram' if report['all_thresholds_passed'] else '✗ FALHA — ver thresholds[*].passed'}")
print(f"  MD5 risks.csv: {checksums.get('risks.csv', '')[:12]}...")
print("=" * 60)
print("Baixe validation_report.json para auditoria local.")

In [ ]:
# ============================================================
# CÉLULA 13c — Empacota output/ para publicação no main
# ============================================================
# Após executar o pipeline, gera um único `output_TIMESTAMP.zip` em
# DIRS["output"]/.. com o conteúdo COMPLETO de output/ (recursivo,
# incluindo quaisquer extras presentes).
#
# Por que isso existe:
# - Colab executa o pipeline e escreve em MyDrive/PTD_Scraper/output/.
# - O caminho CANÔNICO de publicação é o CI (monthly-refresh.yml), que roda
#   o mesmo pipeline e abre PR revisado. Este bundle é o FALLBACK para runs
#   Colab (sem ambiente local, ou gov.br bloqueando IPs de runner) — e
#   também o artefato de transparência: o run inteiro num zip auditável.
# - Dados entram na main por PR, nunca por push direto: commit parcial ou
#   inconsistente é bloqueado pelo CI (notebook-consistency.yml + pytest).
# ============================================================

import os
import zipfile
from datetime import datetime

out_dir = DIRS["output"]
zip_name = f"output_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
zip_path = os.path.join(os.path.dirname(out_dir), zip_name)

# Núcleo obrigatório: o que as CÉLULAS sempre geram no top-level de output/.
# Fora da lista (derivados pós-pipeline, gerados por build_*.py — ver README):
#   - manifest.json (build_manifest.py), coverage_summary.csv (build_coverage.py)
#   - datapackage.json, metadata/, harmonized/, variations.csv, PTD-corpus.xlsx
#     (build_metadata / build_corpus / build_variations / build_xlsx)
EXPECTED_OUTPUTS = [
    "validation_report.json",
    "pdf_metadata.csv",
    "risks.csv",
    "risks.json",
    "deliveries.csv",
    "deliveries.json",
    "organs.csv",
    "error_report.csv",
    "vocabulary_mapping.csv",
]

_missing = [f for f in EXPECTED_OUTPUTS
            if not os.path.exists(os.path.join(out_dir, f))]
if _missing:
    raise RuntimeError(
        f"Pipeline incompleto — {len(_missing)} artefato(s) essenciais "
        f"ausentes em output/: {', '.join(_missing)}. Execute as células "
        f"anteriores (04b→13b) antes de publicar.")

# Zip recursivo de TODO o output/. Arcnames sempre sob 'output/' por
# construção (relpath ancorado em out_dir) — sem path traversal.
_entries = []
for _root, _dirs, _files in os.walk(out_dir):
    _dirs.sort()
    for _fname in sorted(_files):
        _fpath = os.path.join(_root, _fname)
        _arc = os.path.join("output", os.path.relpath(_fpath, out_dir))
        _entries.append((_fpath, _arc))

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for _fpath, _arc in _entries:
        zf.write(_fpath, arcname=_arc)

print("=" * 60)
print("BUNDLE DE PUBLICAÇÃO")
print("=" * 60)
print(f"  Arquivo: {zip_path}")
print(f"  Conteúdo: {len(_entries)} arquivos (output/ completo)")
for _fpath, _arc in _entries:
    print(f"    {_arc:<48s} {os.path.getsize(_fpath) / 1024:>8.1f} KB")
print("\nPara publicar (fallback — o caminho canônico é o CI mensal, que abre PR sozinho):")
print("  1. Baixe o zip pelo painel de Files do Colab (ou aguarde o download automático)")
print("  2. No seu clone local do repo PTD:")
print(f"       unzip -o {zip_name} && \\")
print("       python build_coverage.py && python build_manifest.py && python build_metadata.py && \\")
print("       python build_corpus.py && python build_variations.py && python build_xlsx.py && \\")
print("       git checkout -b data-refresh/$(date +%Y-%m) && \\")
print("       git add output/ && \\")
print("       git commit -m 'data: refresh output/ — Colab run' && \\")
print("       git push -u origin data-refresh/$(date +%Y-%m)")
print("  3. Abra o PR para main — o CI valida o refresh (testes, checksums,")
print("     derivados); Pages reflete ~1 min após o merge")
print("=" * 60)

# Em ambiente Colab, oferece o download programático.
try:
    from google.colab import files as _gc_files
    print("\nIniciando download automático do bundle…")
    _gc_files.download(zip_path)
except ImportError:
    pass  # headless/local — o zip fica em disco